In [ ]:
%pip install -qU transformers==4.57.1 tiktoken wandb
%pip show transformers

## 環境構築

In [ ]:
# RustBPEのビルドとインストール

%pip install maturin
import os

if not os.path.exists("nanochat"):
    !git clone https://github.com/karpathy/nanochat

try:
    # Google Colabの場合
    from google.colab import userdata

    # RustBPEのビルドとインストール
    # curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y && . "$HOME/.cargo/env" && maturin build --release --manifest-path nanochat/rustbpe/Cargo.toml && pip install nanochat/rustbpe/target/wheels/*.whl

    if not os.path.exists("nanochat/rustbpe/target"):
        raise FileNotFoundError("rustbpeのビルドとインストールをターミナルで実行してください。")

except ImportError:
    # ローカル環境の場合
    !maturin develop --release --manifest-path nanochat/rustbpe/Cargo.toml

import rustbpe

In [ ]:
# ログ設定

import logging as logging

if os.path.exists('debug.log'):
    os.remove('debug.log')

def custom_format(record):
    match record.levelno:
        case logging.DEBUG:
            level = '🟦'
        case logging.INFO:
            level = '🟩'
        case logging.WARNING:
            level = '🟨'
        case logging.ERROR:
            level = '🟥'
        case logging.CRITICAL:
            level = '🛑'
    return f"{level} {record.getMessage()}"

logger = logging.getLogger()

for handler in logger.handlers:
    logger.removeHandler(handler)

formatter = logging.Formatter()
formatter.format = custom_format

file_handler = logging.FileHandler('debug.log')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.setLevel(logging.DEBUG)
logger.debug("ログを初期化")

In [ ]:
# ベースディレクトリを設定

def get_base_dir():
    """
    ベースディレクトリのパスを取得する
    デフォルトは、~/.cache/nanochat
    NANOCHAT_BASE_DIR環境変数で上書き可能

    Returns:
        str: ベースディレクトリのパス
    """
    if os.environ.get("NANOCHAT_BASE_DIR"):
        nanochat_dir = os.environ.get("NANOCHAT_BASE_DIR")
    else:
        home_dir = os.path.expanduser("~")
        cache_dir = os.path.join(home_dir, ".cache")
        nanochat_dir = os.path.join(cache_dir, "nanochat")
    os.makedirs(nanochat_dir, exist_ok=True)
    return nanochat_dir

base_dir = get_base_dir()
os.makedirs(os.path.join(base_dir, "tokenizer"), exist_ok=True)
# checkpoint_dir = os.path.join(base_dir, "base_checkpoints", "d20")
os.makedirs(os.path.join(base_dir, "base_checkpoints", "d20"), exist_ok=True)
logger.debug(f"ベースディレクトリ: {base_dir}")

In [ ]:
from filelock import FileLock
import urllib

# BASE_URL = "https://huggingface.co/datasets/karpathy/fineweb-edu-100b-shuffle/resolve/main"
BASE_URL = "https://huggingface.co/nanochat-students/base-d20/resolve/main"

def download_file_with_lock(url, filename, postprocess_fn=None):
    """
    Downloads a file from a URL to a local path in the base directory.
    Uses a lock file to prevent concurrent downloads among multiple ranks.
    """
    base_dir = get_base_dir()
    file_path = os.path.join(base_dir, filename)
    lock_path = file_path + ".lock"

    if os.path.exists(file_path):
        return file_path

    with FileLock(lock_path):
        # Only a single rank can acquire this lock
        # All other ranks block until it is released

        # Recheck after acquiring lock
        if os.path.exists(file_path):
            return file_path

        # Download the content as bytes
        print(f"Downloading {url}...")
        with urllib.request.urlopen(url) as response:
            content = response.read() # bytes

        # Write to local file
        with open(file_path, 'wb') as f:
            f.write(content)
        print(f"Downloaded to {file_path}")

        # Run the postprocess function if provided
        if postprocess_fn is not None:
            postprocess_fn(file_path)

    return file_path

download_file_with_lock(f"{BASE_URL}/token_bytes.pt", "tokenizer/token_bytes.pt")
download_file_with_lock(f"{BASE_URL}/tokenizer.pkl", "tokenizer/tokenizer.pkl")
download_file_with_lock(f"{BASE_URL}/model_021400.pt", "base_checkpoints/d20/model_021400.pt")
download_file_with_lock(f"{BASE_URL}/optim_021400.pt", "base_checkpoints/d20/optim_021400.pt")
download_file_with_lock(f"{BASE_URL}/meta_021400.json", "base_checkpoints/d20/meta_021400.json")

### トークナイザーのロード

In [ ]:
import pickle
import rustbpe
import tiktoken
import os
import copy
from functools import lru_cache

In [ ]:
# 特殊トークン

SPECIAL_TOKENS = [
    # 事前学習で使用
    "<|bos|>", # 文の開始
    # ファインチューニング時に使用
    "<|user_start|>", # ユーザーメッセージ
    "<|user_end|>",
    "<|assistant_start|>", # アシスタントメッセージ
    "<|assistant_end|>",
    "<|python_start|>", # アシスタントがPython REPLツールを呼び出す
    "<|python_end|>",
    "<|output_start|>", # Python REPLがアシスタントに出力を返す
    "<|output_end|>",
]

logger.debug(f"特殊トークン数 {len(SPECIAL_TOKENS)=}")

In [ ]:
class RustBPETokenizer:
    """
    rustbpeのラッパークラス
    訓練時はrustbpeを使用し、推論時はtiktokenを使用する
    """

    def __init__(self, enc, bos_token):
        logger.debug(f"RustBPETokenizer初期化開始 {enc=} {bos_token=}")

        # tiktokenのEncodingオブジェクト
        self.enc = enc 

        # BOSトークンIDをキャッシュ
        self.bos_token_id = self.encode_special(bos_token)
        logger.debug(f"RustBPETokenizer初期化完了 {self.bos_token_id=}")

    @classmethod # クラスメソッドとして定義
    def train_from_iterator(cls, text_iterator, vocab_size):
        """
        rustbpeでトークナイザーを訓練し、tiktokenのエンコーダーを構築する

        Args:
            text_iterator (iterable): テキストのイテレータ
            vocab_size (int): 語彙数
        Returns:
            RustBPETokenizer: 訓練済みのRustBPETokenizerオブジェクト
        """

        logger.debug(f"RustBPETokenizerの訓練開始 {vocab_size=}")

        # 1) rustbpeを訓練

        # rustbpeのTokenizerオブジェクトを作成
        tokenizer = rustbpe.Tokenizer()

        # 特殊トークンは後で__init__で挿入されるため、ここでは訓練しない
        vocab_size_no_special = vocab_size - len(SPECIAL_TOKENS)

        assert vocab_size_no_special >= 256, f"vocab_size_no_special must be at least 256, got {vocab_size_no_special}"

        # トークナイザーを訓練
        tokenizer.train_from_iterator(text_iterator, vocab_size_no_special, pattern=SPLIT_PATTERN)

        # 2) tiktokenのエンコーダーを構築

        # 事前トークン化の正規表現パターン
        pattern = tokenizer.get_pattern()

        # tiktokenに対応したマージルールを作成
        # {バイト列: マージの優先順位ランク}
        mergeable_ranks_list = tokenizer.get_mergeable_ranks()
        mergeable_ranks = {bytes(k): v for k, v in mergeable_ranks_list}

        # tiktokenに対応した特殊トークンの辞書を作成
        # {トークン名: トークンID}
        tokens_offset = len(mergeable_ranks)
        special_tokens = {name: tokens_offset + i for i, name in enumerate(SPECIAL_TOKENS)}

        # tiktokenのエンコーダーを構築
        enc = tiktoken.Encoding(
            name="rustbpe",
            pat_str=pattern,
            mergeable_ranks=mergeable_ranks,
            special_tokens=special_tokens,
        )

        logger.debug(f"RustBPETokenizerの訓練完了")

        # RustBPETokenizerオブジェクトを返す
        # clsでこのクラスのコンストラクタが呼ばれる
        return cls(enc, "<|bos|>")

    @classmethod
    def from_directory(cls, tokenizer_dir):
        """
        訓練済みのRustBPETokenizerをディレクトリから読み込む

        Args:
            tokenizer_dir (str): トークナイザーの保存ディレクトリ
        Returns:
            RustBPETokenizer: 読み込んだRustBPETokenizerオブジェクト
        """

        logger.debug(f"RustBPETokenizerをディレクトリから読み込み開始 {tokenizer_dir=}")

        pickle_path = os.path.join(tokenizer_dir, "tokenizer.pkl")
        logger.debug(f"トークナイザーファイルパス {pickle_path=}")

        with open(pickle_path, "rb") as f:
            enc = pickle.load(f)

        logger.debug(f"RustBPETokenizerをディレクトリから読み込み完了 {enc=}")
        return cls(enc, "<|bos|>")

    @classmethod
    def from_pretrained(cls, tiktoken_name):
        """
        学習済みのtiktokenトークナイザーを読み込む

        Args:
            tiktoken_name (str): tiktokenのエンコーディング名
        Returns:
            RustBPETokenizer: 読み込んだRustBPETokenizerオブジェクト
        """
        logger.debug(f"学習済みのtiktokenトークナイザーを読み込み開始 {tiktoken_name=}")

        # https://github.com/openai/tiktoken/blob/eedc8563/tiktoken_ext/openai_public.py
        enc = tiktoken.get_encoding(tiktoken_name)

        # nanochatでは<|bos|>を使用するが、tiktokenのgpt2などでは<|endoftext|>が使用されているため

        logger.debug(f"学習済みのtiktokenトークナイザーを読み込み完了 {enc=}")
        return cls(enc, "<|endoftext|>")

    def get_vocab_size(self):
        return self.enc.n_vocab

    def get_special_tokens(self):
        return self.enc.special_tokens_set

    def id_to_token(self, id):
        return self.enc.decode([id])

    @lru_cache(maxsize=32)
    def encode_special(self, text):
        return self.enc.encode_single_token(text)

    def get_bos_token_id(self):
        return self.bos_token_id

    def encode(self, text, prepend=None, append=None, num_threads=8):
        """
        テキストをトークンIDにエンコードする

        Args:
            text (str or list[str]): エンコードするテキストまたはテキストのリスト
            prepend (str or int, optional): 先頭に追加する特殊トークン（文字列またはトークンID）
            append (str or int, optional): 末尾に追加する特殊トークン（文字列またはトークンID）
            num_threads (int, optional): バッチエンコード時のスレッド数（デフォルト: 8）
        Returns:
            list[int] or list[list[int]]: トークンIDのリストまたはトークンIDのリストのリスト
        """
        logger.debug(f"エンコード開始 {len(text)=} {prepend=} {append=} {num_threads=}")

        # 1) 特殊トークンの設定

        # 出力の先頭に付ける特殊トークンIDを取得
        # BOSトークン
        if prepend is not None:
            prepend_id = prepend if isinstance(prepend, int) else self.encode_special(prepend)
            logger.debug(f"{prepend_id=}") # 65527

        # 出力の末尾に付ける特殊トークンIDを取得
        # なし
        if append is not None:
            append_id = append if isinstance(append, int) else self.encode_special(append)
            logger.debug(f"{append_id=}") 

        # 2) エンコード

        # 入力が文字列の場合
        if isinstance(text, str):

            # tiktokenでエンコード
            ids = self.enc.encode_ordinary(text)

            if prepend is not None:
                # 先頭に特殊トークンを追加
                ids.insert(0, prepend_id)

            if append is not None:
                # 末尾に特殊トークンを追加
                ids.append(append_id)

        # 入力が文字列のリストの場合
        elif isinstance(text, list):

            # tiktokenでバッチエンコード
            ids = self.enc.encode_ordinary_batch(text, num_threads=num_threads)

            if prepend is not None:
                # 先頭に特殊トークンを追加
                for ids_row in ids:
                    ids_row.insert(0, prepend_id)

            if append is not None:
                # 末尾に特殊トークンを追加
                for ids_row in ids:
                    ids_row.append(append_id)
        else:
            raise ValueError(f"Invalid input type: {type(text)}")

        logger.debug(f"エンコード完了 {len(ids)=}")
        return ids

    def __call__(self, *args, **kwargs):
        return self.encode(*args, **kwargs)

    def decode(self, ids):
        """
        トークンIDのリストをデコードしてテキストに変換する

        Args:
            ids (list[int]): トークンIDのリスト
        Returns:
            str: デコードしたテキスト
        """
        logger.debug(f"デコード開始 {len(ids)=}")

        # tiktokenでデコード
        res = self.enc.decode(ids)

        logger.debug(f"デコード完了 {len(res)=}")
        return res

    def save(self, tokenizer_dir):
        """
        訓練後のトークナイザーをディレクトリに保存する

        Args:
            tokenizer_dir (str): トークナイザーの保存ディレクトリ
        """
        logger.debug(f"トークナイザーの保存開始 {tokenizer_dir=}")

        # 1) ディレクトリを作成

        os.makedirs(tokenizer_dir, exist_ok=True)
        pickle_path = os.path.join(tokenizer_dir, "tokenizer.pkl")
        logger.debug(f"トークナイザーファイルパス {pickle_path=}")

        # 2) pickleで保存

        with open(pickle_path, "wb") as f:
            pickle.dump(self.enc, f)

        logger.debug(f"トークナイザーの保存完了 {pickle_path=}")

    def render_conversation(self, conversation, max_tokens=2048):
        """
        チャット形式のデータをトークン化する
        事前学習では使わない

        Args:
            conversation: dict チャット会話データ
            max_tokens: int 最大トークン数
        Returns:
            ids: list[int] トークンIDのリスト
            mask: list[int] 同じ長さのマスクリスト、mask=1はアシスタントが学習すべきトークン
        """
        logger.debug(f"会話のレンダリング開始 {conversation=} {max_tokens=}")

        # ids, masks that we will return and a helper function to help build them up.
        ids, mask = [], []

        def add_tokens(token_ids, mask_val):
            if isinstance(token_ids, int):
                token_ids = [token_ids]
            ids.extend(token_ids)
            mask.extend([mask_val] * len(token_ids))

        # sometimes the first message is a system message...
        # => just merge it with the second (user) message
        if conversation["messages"][0]["role"] == "system":
            # some conversation surgery is necessary here for now...
            conversation = copy.deepcopy(conversation) # avoid mutating the original
            messages = conversation["messages"]
            assert messages[1]["role"] == "user", "System message must be followed by a user message"
            messages[1]["content"] = messages[0]["content"] + "\n\n" + messages[1]["content"]
            messages = messages[1:]
        else:
            messages = conversation["messages"]

        assert len(messages) >= 1, f"Conversation has less than 1 message: {messages}"

        # fetch all the special tokens we need
        bos = self.get_bos_token_id()
        user_start, user_end = self.encode_special("<|user_start|>"), self.encode_special("<|user_end|>")
        assistant_start, assistant_end = self.encode_special("<|assistant_start|>"), self.encode_special("<|assistant_end|>")
        python_start, python_end = self.encode_special("<|python_start|>"), self.encode_special("<|python_end|>")
        output_start, output_end = self.encode_special("<|output_start|>"), self.encode_special("<|output_end|>")

        # 開始トークンを追加
        add_tokens(bos, 0)

        # メッセージを順に処理
        for i, message in enumerate(messages):

            # some sanity checking here around assumptions, to prevent footguns
            must_be_from = "user" if i % 2 == 0 else "assistant"
            assert message["role"] == must_be_from, f"Message {i} is from {message['role']} but should be from {must_be_from}"

            # content can be either a simple string or a list of parts (e.g. containing tool calls)
            content = message["content"]

            # ユーザーメッセージの場合
            if message["role"] == "user":
                assert isinstance(content, str), "User messages are simply expected to be strings"
                value_ids = self.encode(content)

                # 損失計算に含めない
                add_tokens(user_start, 0)
                add_tokens(value_ids, 0)
                add_tokens(user_end, 0)

            # アシスタントメッセージの場合
            elif message["role"] == "assistant":

                # アシスタントの開始トークンを追加
                add_tokens(assistant_start, 0)

                if isinstance(content, str):
                    # テキストをトークン化
                    value_ids = self.encode(content)

                    # 損失計算に含める
                    add_tokens(value_ids, 1)

                elif isinstance(content, list):
                    for part in content:
                        # テキストをトークン化
                        value_ids = self.encode(part["text"])

                        # 通常のテキストの場合
                        if part["type"] == "text":

                            # 損失計算に含める
                            add_tokens(value_ids, 1)

                        # Pythonのツール呼び出しの場合
                        elif part["type"] == "python":
                            # ツールの開始と終了トークンを追加
                            add_tokens(python_start, 1)
                            add_tokens(value_ids, 1)
                            add_tokens(python_end, 1)

                        # Pythonの出力の場合
                        elif part["type"] == "python_output":
                            # ツールの出力の開始と終了トークンを追加
                            # 推論時はPythonの出力を用いるため損失計算に含めない
                            add_tokens(output_start, 0)
                            add_tokens(value_ids, 0)
                            add_tokens(output_end, 0)
                        else:
                            raise ValueError(f"Unknown part type: {part['type']}")
                else:
                    raise ValueError(f"Unknown content type: {type(content)}")
                add_tokens(assistant_end, 1)

        # 最大トークン数に切り詰める
        ids = ids[:max_tokens]
        mask = mask[:max_tokens]

        logger.debug(f"会話のレンダリング完了 {ids=} {mask=}")
        return ids, mask

    def visualize_tokenization(self, ids, mask, with_token_id=False):
        """
        render_conversationの結果の可視化
        今回は使わない
        """
        RED = '\033[91m'
        GREEN = '\033[92m'
        RESET = '\033[0m'
        GRAY = '\033[90m'
        tokens = []
        for i, (token_id, mask_val) in enumerate(zip(ids, mask)):
            token_str = self.decode([token_id])
            color = GREEN if mask_val == 1 else RED
            tokens.append(f"{color}{token_str}{RESET}")
            if with_token_id:
                tokens.append(f"{GRAY}({token_id}){RESET}")
        return '|'.join(tokens)

    def render_for_completion(self, conversation):
        """
        Assistantの補完を促すために会話をレンダリングする
        今回は使わない
        """
        # We have some surgery to do: we need to pop the last message (of the Assistant)
        conversation = copy.deepcopy(conversation) # avoid mutating the original
        messages = conversation["messages"]
        assert messages[-1]["role"] == "assistant", "Last message must be from the Assistant"
        messages.pop() # remove the last message (of the Assistant) inplace

        # Now tokenize the conversation
        ids, mask = self.render_conversation(conversation)

        # Finally, to prime the Assistant for a completion, append the Assistant start token
        assistant_start = self.encode_special("<|assistant_start|>")
        ids.append(assistant_start)
        return ids

In [ ]:
# 事前トークン化の正規表現を設定

# GPT-4と同じ
SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

logger.debug(f"事前トークン化の正規表現: {SPLIT_PATTERN=}")

In [ ]:
def get_tokenizer():
    """
    RustBPETokenizerオブジェクトを返す

    Returns:
        RustBPETokenizer: トークナイザーオブジェクト
    """
    base_dir = get_base_dir()
    tokenizer_dir = os.path.join(base_dir, "tokenizer")
    # return HuggingFaceTokenizer.from_directory(tokenizer_dir)
    return RustBPETokenizer.from_directory(tokenizer_dir)

tokenizer = get_tokenizer()

### モデルのロード

In [ ]:
import math
from functools import partial
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def is_ddp():
    """
    分散データ並列（Distributed Data Parallel, DDP）が有効かどうか

    Returns:
        bool: DDPが有効な場合True、そうでない場合False
    """

    # RANK環境変数はtorchrunなどによって設定されるプロセスの識別子
    return int(os.environ.get('RANK', -1)) != -1

is_ddp()

In [ ]:
def get_dist_info():
    """
    分散データ並列（Distributed Data Parallel, DDP）訓練の環境変数を取得
    """
    if is_ddp():
        assert all(var in os.environ for var in ['RANK', 'LOCAL_RANK', 'WORLD_SIZE'])

        # グローバルなプロセス識別子
        ddp_rank = int(os.environ['RANK'])

        # ローカル（同一ノード内）のプロセス識別子
        ddp_local_rank = int(os.environ['LOCAL_RANK'])

        # ワールドサイズ（全プロセス数）
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        return True, ddp_rank, ddp_local_rank, ddp_world_size
    else:
        return False, 0, 0, 1

get_dist_info()

In [ ]:
def norm(x):
    """
    平方根平均二乗ノルム正規化（Root Mean Square Normalization, RMSNorm）を適用する

    Args:
        x (Tensor): 入力テンソル
    Returns:
        Tensor: 正規化されたテンソル
    """
    logger.debug(f"RMSNormを適用開始 {x.shape=}")

    # ゲインやバイアスを持たないため高速
    # (4, 2048, 1280) -> (4, 2048, 1280)
    res = F.rms_norm(x, (x.size(-1),))

    logger.debug(f"RMSNorm適用完了 {res.shape=}")
    return res

In [ ]:
def apply_rotary_emb(x, cos, sin):
    """
    回転位置埋め込み（RoPE, Rotary Positional Embedding）を適用する

    Args:
        x (Tensor): 入力テンソル
        cos (Tensor): コサイン成分、形状は(1, seq_len, 1, head_dim/2)
        sin (Tensor): サイン成分、形状は(1, seq_len, 1, head_dim/2)
    Returns:
        Tensor: RoPEが適用されたテンソル、形状は(batch_size, seq_len, num_heads, head_dim)
    """
    logger.debug(f"RoPEを適用 {x.shape=}, {x.dtype=} {cos.shape=}, {cos.dtype=} {sin.shape=} {sin.dtype=}")

    assert x.ndim == 4

    # head_dimを2で分割
    d = x.shape[3] // 2

    # 最後の次元を2つに分割
    x1, x2 = x[..., :d], x[..., d:]

    # 2次元回転行列を適用
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    out = torch.cat([y1, y2], 3) # 最後の次元で結合

    # sinとcosはfloat32で計算されることが多いため、元のデータ型に戻す
    out = out.to(x.dtype)

    logger.debug(f"RoPE適用完了 {out.shape=}")
    return out

In [ ]:
@dataclass
class GPTConfig:
    sequence_len: int = 1024 # 最大シーケンス長
    vocab_size: int = 50304 # 語彙サイズ
    n_layer: int = 12 # Transformerのレイヤー数
    n_head: int = 6 # アテンションヘッド数
    n_kv_head: int = 6 # キーバリューヘッド数
    n_embd: int = 768 # Transformerの埋め込み次元数

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    GQA（Group Query Attention）を実装した因果セルフアテンションモジュール
    """

    def __init__(self, config, layer_idx):
        logger.debug(f"CausalSelfAttentionを初期化開始 {config.n_head=} {config.n_kv_head=} {config.n_embd=} {layer_idx=}")

        super().__init__()

        self.layer_idx = layer_idx

        # 10
        self.n_head = config.n_head

        # 10
        self.n_kv_head = config.n_kv_head

        # 1280
        self.n_embd = config.n_embd

        # 1280 / 10 = 128
        self.head_dim = self.n_embd // self.n_head
        logger.debug(f"{self.head_dim=}")

        assert self.n_embd % self.n_head == 0
        assert self.n_kv_head <= self.n_head and self.n_head % self.n_kv_head == 0

        # 1280 -> 1280
        self.c_q = nn.Linear(self.n_embd, self.n_head * self.head_dim, bias=False)

        # 1280 -> 1280
        self.c_k = nn.Linear(self.n_embd, self.n_kv_head * self.head_dim, bias=False)

        # 1280 -> 1280
        self.c_v = nn.Linear(self.n_embd, self.n_kv_head * self.head_dim, bias=False)

        # 1280 -> 1280
        self.c_proj = nn.Linear(self.n_embd, self.n_embd, bias=False)

        logger.debug("CausalSelfAttentionの初期化完了")

    def forward(self, x, cos_sin, kv_cache):
        logger.debug(f"CausalSelfAttentionの順伝搬を実行 {x.shape=} {x.dtype=} {cos_sin[0].shape=} {cos_sin[0].dtype=} {kv_cache if kv_cache is not None else None=}")

        # (4, 2048, 1280)
        B, T, C = x.size()

        # 1) クエリ・キー・バリューを計算

        # (4, 2048, 1280) -> (4, 2048, 1280) -> (4, 2048, 10, 128)
        q = self.c_q(x).view(B, T, self.n_head, self.head_dim)

        # (4, 2048, 1280) -> (4, 2048, 1280) -> (4, 2048, 10, 128)
        k = self.c_k(x).view(B, T, self.n_kv_head, self.head_dim)

        # (4, 2048, 1280) -> (4, 2048, 1280) -> (4, 2048, 10, 128)
        v = self.c_v(x).view(B, T, self.n_kv_head, self.head_dim)

        # 2) クエリとキーにRoPEを適用

        # (1, 2048, 1, 64), (1, 2048, 1, 64)
        cos, sin = cos_sin

        # (4, 2048, 10, 128), (4, 2048, 10, 128)
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)

        # 3) RMSNormを適用

        # (4, 2048, 10, 128) -> (4, 2048, 10, 128)
        # (4, 2048, 10, 128) -> (4, 2048, 10, 128)
        q, k = norm(q), norm(k)

        # 4) KVキャッシュを適用

        # (4, 2048, 10, 128) -> (4, 10, 2048, 128)
        # (4, 2048, 10, 128) -> (4, 10, 2048, 128)
        # (4, 2048, 10, 128) -> (4, 10, 2048, 128)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        # KVキャッシュがある場合（推論時）
        if kv_cache is not None:
            # KVキャッシュを更新し、これまでの全てのKVを取得
            k, v = kv_cache.insert_kv(self.layer_idx, k, v)
            logger.debug(f"KVキャッシュを更新 {k.shape=} {v.shape=}")

        # 5) スケールド・ドットプロダクト・アテンションを計算

        # クエリの数
        # 2048
        Tq = q.size(2)
        logger.debug(f"{Tq=}")

        # キー/バリューの数（キャッシュ内も含む）
        # 2048
        Tk = k.size(2)
        logger.debug(f"{Tk=}")

        # GQAを有効にするかどうか
        # False
        enable_gqa = self.n_head != self.n_kv_head
        logger.debug(f"GQAを有効化: {enable_gqa=}")

        # 訓練時またはクエリ数とキー/バリュー数が等しい場合
        if kv_cache is None or Tq == Tk:
            logger.debug("通常の因果アテンションを適用")
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=enable_gqa)

        # 推論時でクエリが1つだけの場合
        elif Tq == 1:
            logger.debug("マスクを使用しない因果アテンションを適用")
            y = F.scaled_dot_product_attention(q, k, v, is_causal=False, enable_gqa=enable_gqa)

        # 推論時で複数のトークンを一度に処理する場合（prefill）
        else:
            logger.debug("マスクを手動で作成して因果アテンションを適用")

            # アテンションマスクを初期化
            attn_mask = torch.zeros((Tq, Tk), dtype=torch.bool, device=q.device)
            logger.debug(f"{attn_mask.shape=}")

            # キャッシュ済みの長さを取得
            prefix_len = Tk - Tq
            logger.debug(f"{prefix_len=}")

            if prefix_len > 0:
                # マスクの左側（キャッシュ部分）をマスクしない
                attn_mask[:, :prefix_len] = True

            # マスクの右側に下三角行列を設定
            attn_mask[:, prefix_len:] = torch.tril(
                torch.ones((Tq, Tq), dtype=torch.bool, device=q.device)
            )

            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, enable_gqa=enable_gqa)

        # 6) 出力処理

        # (4, 10, 2048, 128) -> (4, 2048, 10, 128) -> (4, 2048, 1280)
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        logger.debug(f"アテンション出力を整形 {y.shape=}")

        # (4, 2048, 1280) -> (4, 2048, 1280)
        y = self.c_proj(y)
        logger.debug(f"最終線形変換を適用 {y.shape=}")

        logger.debug(f"CausalSelfAttentionの順伝搬完了 {y.shape=}")
        return y

In [ ]:
class MLP(nn.Module):
    """
    フィードフォワードネットワーク（FFN）モジュール
    """

    def __init__(self, config):
        logger.debug(f"MLPを初期化 {config.n_embd=}")
        super().__init__()

        # 1280 -> 5120
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)

        # 5120 -> 1280
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)

        logger.debug("MLPの初期化完了")

    def forward(self, x):
        logger.debug(f"MLPの順伝搬を実行 {x.shape=} {x.dtype=}")

        x = self.c_fc(x)
        logger.debug(f"{x.shape=}")

        # Squared ReLU活性化関数を適用
        # GELUやSiLU（Swish）よりも高速でメモリ効率が良い
        x = F.relu(x).square()

        x = self.c_proj(x)
        logger.debug(f"MLPの順伝搬完了 {x.shape=}")
        return x

In [ ]:
class Block(nn.Module):
    """
    Transformerブロックモジュール
    """

    def __init__(self, config, layer_idx):
        logger.debug(f"Transformer Blockを初期化 {config.n_embd=} {layer_idx=}")
        super().__init__()
        self.attn = CausalSelfAttention(config, layer_idx)
        self.mlp = MLP(config)
        logger.debug("Transformer Blockの初期化完了")

    def forward(self, x, cos_sin, kv_cache):
        logger.debug(f"Transformer Blockの順伝搬を実行 {x.shape=} {x.dtype=} {cos_sin[0].shape=} {cos_sin[0].dtype=} {kv_cache if kv_cache is not None else None=}")

        x = x + self.attn(norm(x), cos_sin, kv_cache)

        x = x + self.mlp(norm(x))

        logger.debug(f"Transformer Blockの順伝搬完了 {x.shape=}")
        return x

In [ ]:
class GPT(nn.Module):
    """
    GPTモデル全体
    """

    def __init__(self, config):
        logger.debug(f"GPTモデルを初期化 {config.n_layer=} {config.n_head=} {config.n_embd=} {config.vocab_size=} {config.sequence_len=}")

        super().__init__()

        self.config = config

        # Transformerのエンコーダー部分
        self.transformer = nn.ModuleDict({
            # 単語埋め込み層
            # 65536 -> 1280
            "wte": nn.Embedding(config.vocab_size, config.n_embd),

            # 20層のTransformerブロックを作成
            "h": nn.ModuleList(
                [Block(config, layer_idx) for layer_idx in range(config.n_layer)]
            ),
        })

        # 出力の線形層（Language Model Head）
        # 1280 -> 65536
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # RoPEのシーケンス長を設定
        # KVキャッシュでシーケンスの最大長を超える可能性があるため余裕をもたせる
        # 2048 * 10 = 20480
        self.rotary_seq_len = config.sequence_len * 10
        logger.debug(f"{self.rotary_seq_len=}")

        # ヘッド次元数を計算
        # 1280 / 10 = 128
        head_dim = config.n_embd // config.n_head
        logger.debug(f"{head_dim=}")

        # RoPEのcosとsinを事前計算
        cos, sin = self._precompute_rotary_embeddings(self.rotary_seq_len, head_dim)

        # cosをバッファに登録（state_dictには保存しない）
        self.register_buffer("cos", cos, persistent=False)

        # sinをバッファに登録（state_dictには保存しない）
        self.register_buffer("sin", sin, persistent=False)

        logger.debug("GPTモデルの初期化完了")

    def init_weights(self):
        """
        モデルの重みを初期化する
        """
        logger.debug("GPTモデルの重みを初期化開始")

        # 全てのモジュールに対して初期化関数を適用
        self.apply(self._init_weights)

        # 出力層の重みをゼロに初期化
        torch.nn.init.zeros_(self.lm_head.weight)

        # 全てのTransformerブロックの出力層の重みをゼロに初期化
        for block in self.transformer.h:
            torch.nn.init.zeros_(block.mlp.c_proj.weight)
            torch.nn.init.zeros_(block.attn.c_proj.weight)

        # RoPEのcosとsinを計算
        head_dim = self.config.n_embd // self.config.n_head
        cos, sin = self._precompute_rotary_embeddings(self.rotary_seq_len, head_dim)
        self.cos, self.sin = cos, sin

        # 埋め込み層の重みをbfloat16にダウンキャストし、メモリ使用量を削減
        if self.transformer.wte.weight.device.type == "cuda":
            self.transformer.wte.to(dtype=torch.bfloat16)

        logger.debug("GPTモデルの重みの初期化完了")

    def _init_weights(self, module):
        """
        init_weightsから呼び出される重み初期化関数

        Args:
            module (nn.Module): 初期化するモジュール
        """
        logger.debug(f"モジュールの重みを初期化開始 {module.__class__.__name__=}")

        # 全結合層の場合
        if isinstance(module, nn.Linear):
            # 論文に基づく方法で重みを初期化
            # https://arxiv.org/pdf/2310.17813
            fan_out = module.weight.size(0)
            fan_in = module.weight.size(1)
            std = 1.0 / math.sqrt(fan_in) * min(1.0, math.sqrt(fan_out / fan_in))
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        # 埋め込み層の場合
        elif isinstance(module, nn.Embedding):
            # 標準正規分布で重みを初期化
            torch.nn.init.normal_(module.weight, mean=0.0, std=1.0)

        logger.debug("モジュールの重みの初期化完了")

    def _precompute_rotary_embeddings(self, seq_len, head_dim, base=10000, device=None):
        """
        RoPEのcosとsinを事前計算する

        Args:
            seq_len (int): シーケンス長
            head_dim (int): ヘッド次元数
            base (int, optional): 逆周波数の基底値。デフォルトは10000。
            device (torch.device, optional): 計算に使用するデバイス。デフォルトはNoneで、自動検出される。
        Returns:
            Tuple[Tensor, Tensor]: 事前計算されたcosとsinのテンソル
        """

        logger.debug(f"RoPEの事前計算を実行 {seq_len=} {head_dim=} {base=}")

        # デバイスが指定されていない場合
        if device is None:
            # 自動検出
            device = self.transformer.wte.weight.device

        # 要素数を半分にするための偶数インデックスを作成
        channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32, device=device)

        # 逆周波数（invert frequency）を計算
        # 角速度と実質的に同じ
        # \theta = base^{-\frac{2i}{head\_dim}}
        # (64,)
        inv_freq = 1.0 / (base ** (channel_range / head_dim))
        logger.debug(f"{inv_freq.shape=}")

        # 文字位置のインデックスを作成
        # (20480,)
        t = torch.arange(seq_len, dtype=torch.float32, device=device)
        logger.debug(f"{t.shape=}")
    
        # 回転角度を計算
        # 回転角度 = 文字位置 * 逆周波数
        # (20480, 64)
        freqs = torch.outer(t, inv_freq)
        logger.debug(f"{freqs.shape=}")

        # 回転角度からcosとsinを計算
        # (20480, 64), (20480, 64)
        cos, sin = freqs.cos(), freqs.sin()

        # bfloat16に変換
        cos, sin = cos.bfloat16(), sin.bfloat16()

        # ブロードキャストのために次元を追加
        # (1, 20480, 1, 64), (1, 20480, 1, 64)
        cos, sin = cos[None, :, None, :], sin[None, :, None, :]

        logger.debug(f"RoPEの事前計算完了 {cos.shape=} {sin.shape=}")
        return cos, sin

    def get_device(self):
        return self.transformer.wte.weight.device

    def estimate_flops(self):
        """
        モデルのトークンあたりのFLOPsを推定する
        FLOPSは1トークン生成するために必要な浮動小数点演算回数
        Chinchillaのスケール則で総訓練ステップ数を導くために必要
        https://arxiv.org/abs/2204.02311
        """
        logger.debug("モデルのFLOPsを推定開始")
    
        # モデルのパラメータ数を計算
        # 560,988,160 = 5.6億
        nparams = sum(p.numel() for p in self.parameters())
        logger.debug(f"{nparams=:,}")

        # 埋め込み層のパラメータ数を取得
        # 83,886,080 = 8,400万
        nparams_embedding = self.transformer.wte.weight.numel()
        logger.debug(f"{nparams_embedding=:,}")

        # レイヤー数l=20
        # ヘッド数h=10
        # クエリの次元q=128
        # シーケンス長t=2048
        l, h, q, t = self.config.n_layer, self.config.n_head, self.config.n_embd // self.config.n_head, self.config.sequence_len
        logger.debug(f"{l=}, {h=}, {q=}, {t=}")

        # FLOPsを計算
        # 3,491,758,080 = 3.5GFLOPs
        num_flops_per_token = 6 * (nparams - nparams_embedding) + 12 * l * h * q * t

        logger.debug(f"モデルのFLOPsの推定完了 {num_flops_per_token=:,}")
        return num_flops_per_token

    def setup_optimizers(self, unembedding_lr=0.004, embedding_lr=0.2, matrix_lr=0.02, weight_decay=0.0):
        """
        最適化関数を設定する

        Args:
            unembedding_lr (float): 出力層の学習率
            embedding_lr (float): 埋め込み層の学習率
            matrix_lr (float): トランスフォーマーの行列パラメータの学習率
            weight_decay (float): 重み減衰（L2正則化）係数
        Returns:
            List[torch.optim.Optimizer]: 設定された最適化関数のリスト
        """
        logger.debug(f"最適化関数を設定開始 {unembedding_lr=} {embedding_lr=} {matrix_lr=} {weight_decay=}")

        # 1) モデルの次元数を取得

        model_dim = self.config.n_embd

        ddp, rank, local_rank, world_size = get_dist_info()

        # トランスフォーマーの行列パラメータ
        matrix_params = list(self.transformer.h.parameters())
        logger.debug(f"{len(matrix_params)=}")

        # トランスフォーマーの埋め込みパラメータ
        embedding_params = list(self.transformer.wte.parameters())

        # 出力層のパラメータ
        lm_head_params = list(self.lm_head.parameters())

        assert len(list(self.parameters())) == len(matrix_params) + len(embedding_params) + len(lm_head_params)

        # 2) 入力の埋め込み層と出力の線形層に対してAdamWを初期化

        # モデルの次元に基づいて学習率をスケーリング
        dmodel_lr_scale = (model_dim / 768) ** -0.5
        if rank == 0:
            logger.debug(f"Scaling the LR for the AdamW parameters ∝1/√({model_dim}/768) = {dmodel_lr_scale:.6f}")

        adam_groups = [
            dict(params=lm_head_params, lr=unembedding_lr * dmodel_lr_scale),
            dict(params=embedding_params, lr=embedding_lr * dmodel_lr_scale),
        ]

        adamw_kwargs = dict(betas=(0.8, 0.95), eps=1e-10, weight_decay=weight_decay)

        AdamWFactory = DistAdamW if ddp else partial(torch.optim.AdamW, fused=True)

        adamw_optimizer = AdamWFactory(adam_groups, **adamw_kwargs)

        # 3) Transformerの行列パラメータに対してMuonを初期化

        muon_kwargs = dict(lr=matrix_lr, momentum=0.95)

        MuonFactory = DistMuon if ddp else Muon

        muon_optimizer = MuonFactory(matrix_params, **muon_kwargs)


        # 4) 2つの最適化関数を1つのリストにまとめる

        optimizers = [adamw_optimizer, muon_optimizer]

        for opt in optimizers:
            for group in opt.param_groups:
                group["initial_lr"] = group["lr"]

        logger.debug("最適化関数の設定完了")
        return optimizers

    def forward(self, idx, targets=None, kv_cache=None, loss_reduction='mean'):
        """
        GPTモデルの順伝搬

        Args:
            idx (Tensor): トークンIDのテンソル、形状は(batch_size, seq_len)
            targets (Tensor, optional): 目標トークンIDのテンソル、形状は(batch_size, seq_len)。デフォルトはNone。
            kv_cache (KVCache, optional): KVキャッシュオブジェクト。デフォルトはNone。
            loss_reduction (str, optional): 損失の集約方法。'mean'または'sum'。デフォルトは'mean'。
        Returns:
            Tensor: 訓練モードの場合は損失テンソル、推論モードの場合はロジットテンソル
            形状は(batch_size, seq_len, vocab_size)
        """

        logger.debug(f"GPTモデルの順伝搬を実行 {idx.shape=} {idx.dtype=} {targets.shape if targets is not None else None=} {kv_cache if kv_cache is not None else None=}")

        B, T = idx.size()

        # 1) RoPEの準備

        assert T <= self.cos.size(1), f"Sequence length grew beyond the rotary embeddings cache: {T} > {self.cos.size(1)}"

        assert idx.device == self.cos.device, f"Rotary embeddings and idx are on different devices: {idx.device} != {self.cos.device}"

        assert self.cos.dtype == torch.bfloat16, "Rotary embeddings must be in bfloat16"

        # KVキャッシュが存在する場合、現在のキャッシュ内の位置にRoPEをオフセットする必要がある
        T0 = 0 if kv_cache is None else kv_cache.get_pos()

        # 事前計算されたRoPEを現在のシーケンス長に切り詰める
        # (1, 2048, 1, 64), (1, 2048, 1, 64)
        cos_sin = self.cos[:, T0:T0+T], self.sin[:, T0:T0+T]
        logger.debug(f"RoPEの切り出し完了 {cos_sin[0].shape=} {cos_sin[1].shape=}")

        # 2) 20層のTransformerの順伝播

        # トークンIDを埋め込みに変換
        x = self.transformer.wte(idx)

        # 最初のRMSNormを適用（Pre-LN）
        x = norm(x)

        # 各Transformerブロックを順番に適用
        for block in self.transformer.h:
            x = block(x, cos_sin, kv_cache)

        # 3) 出力の線形層を適用

        x = norm(x)

        # Logit Softcapping
        # ロジットの値が+/-15を超えないように制限する
        softcap = 15

        # 訓練モードの場合
        if targets is not None:

            # 出力の線形層を適用
            logits = self.lm_head(x)

            # logitsにソフトキャップを適用
            logits = softcap * torch.tanh(logits / softcap)

            # logitsをfloat32にアップキャスト
            logits = logits.float()

            # クロスエントロピー損失を計算
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
                reduction=loss_reduction
            )

            logger.debug(f"GPTモデルの順伝搬完了（訓練モード） {loss.shape=}")
            return loss

        # 推論モードの場合
        else:
            # 出力の線形層を適用
            logits = self.lm_head(x)

            # logitsにソフトキャップを適用
            logits = softcap * torch.tanh(logits / softcap)

            logger.debug(f"GPTモデルの順伝搬完了（推論モード） {logits.shape=}")
            return logits

    @torch.inference_mode() # 勾配計算を無効化
    def generate(self, tokens, max_tokens, temperature=1.0, top_k=None, seed=42):
        """
        テキストを生成するジェネレーター関数

        Args:
            tokens (List[int]): 初期トークンのリスト
            max_tokens (int): 生成する最大トークン数
            temperature (float, optional): 温度パラメータ。デフォルトは1.0。
            top_k (int, optional): Top-KサンプリングのK値。デフォルトはNoneで無効。
            seed (int, optional): 乱数シード。デフォルトは42。
        Yields:
            int: 生成された各トークンID
        """
        logger.debug(f"テキスト生成を開始 {tokens=} {max_tokens=} {temperature=} {top_k=} {seed=}")

        # 1) 初期化

        assert isinstance(tokens, list)

        device = self.get_device()
        logger.debug(f"{device=}")

        rng = None

        # 温度が設定されている場合、乱数生成器を初期化
        if temperature > 0:
            rng = torch.Generator(device=device)
            rng.manual_seed(seed)

        # バッチ次元を追加
        ids = torch.tensor([tokens], dtype=torch.long, device=device)

        # 2) トークン生成ループ

        # 最大トークン数まで生成
        for _ in range(max_tokens):

            # 1トークンを生成
            # (B, T, vocab_size)
            logits = self.forward(ids)

            # 直近のトークンのロジットを取得
            # (B, vocab_size)
            logits = logits[:, -1, :]

            # Top-Kサンプリングが有効な場合
            if top_k is not None:
                # 上位K個以外のロジットを無限小に設定
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            # 温度が設定されている場合
            if temperature > 0:
                # 確率分布をなめらかにしランダムに1つ選択
                logits = logits / temperature
                probs = F.softmax(logits, dim=-1)
                next_ids = torch.multinomial(probs, num_samples=1, generator=rng)

            # 温度が0の場合
            else:
                # 最も高い確率のトークンを選択（Greedy Decoding）
                next_ids = torch.argmax(logits, dim=-1, keepdim=True)

            # 生成したトークンをシーケンスの末尾に追加
            ids = torch.cat((ids, next_ids), dim=1)

            # 生成したトークンをCPUに移動してPythonのintに変換
            token = next_ids.item()

            logger.debug(f"生成トークン: {token}")
            yield token

### 推論エンジン

In [ ]:
class KVCache:
    """
    推論を高速化するためのキー・バリューキャッシュ（KVキャッシュ）モジュール
    """

    def __init__(self, batch_size, num_heads, seq_len, head_dim, num_layers):
        logger.debug(f"KVCacheを初期化開始 {batch_size=} {num_heads=} {seq_len=} {head_dim=} {num_layers=}")

        # モデルの全層にKVキャッシュがある
        self.kv_shape = (num_layers, 2, batch_size, num_heads, seq_len, head_dim)

        self.kv_cache = None

        # キャッシュ内の書き込み位置
        self.pos = 0

        logger.debug("KVCacheの初期化完了")

    def reset(self):
        self.pos = 0

    def get_pos(self):
        return self.pos

    def prefill(self, other):
        """
        KVキャッシュを別のKVキャッシュで事前設定（prefill）する
        オプションでバッチ次元に沿って拡張することも可能
        複数のサンプルを並列に生成する場合に使用

        Args:
            other (KVCache): 事前設定に使用する別のKVキャッシュ
        """
        logger.debug(f"KVキャッシュを事前設定開始 {other.kv_cache is not None=}")

        # 1) 形状を検証

        assert self.kv_cache is None, "KVキャッシュが空でない場合、事前設定できません"
        assert other.kv_cache is not None, "事前設定に使用するKVキャッシュが空です"

        # 各次元を検証
        for ix, (dim1, dim2) in enumerate(zip(self.kv_shape, other.kv_shape)):
            # Transformerの層数、K/V、ヘッド数、ヘッド次元の検証
            if ix in [0, 1, 3, 5]:
                assert dim1 == dim2, f"Batch dim mismatch: {dim1} != {dim2}"

            # バッチサイズの検証
            elif ix == 2:
                # バッチサイズは拡張可能
                assert dim1 == dim2 or dim2 == 1, f"Batch dim mismatch: {dim1} != {dim2}"

            # シーケンス長の検証
            elif ix == 4:
                # シーケンス長は長い必要がある
                assert dim1 >= dim2, f"Seq len mismatch: {dim1} < {dim2}"

        # 2) キャッシュを初期化

        dtype, device = other.kv_cache.dtype, other.kv_cache.device
        self.kv_cache = torch.empty(self.kv_shape, dtype=dtype, device=device)

        # 3) データをコピー

        # バッチ次元に沿って複製
        self.kv_cache[:, :, :, :, :other.pos, :] = other.kv_cache

        # 4) 書き込み位置を更新

        self.pos = other.pos

        logger.debug("KVキャッシュの事前設定完了")

    def insert_kv(self, layer_idx, k, v):
        """
        KVキャッシュに新しいキーとバリューを挿入し、これまでの全てのキーとバリューを返す

        Args:
            layer_idx (int): 現在のTransformer層のインデックス
            k (Tensor): 新しいキーのテンソル、形状は (B, H, T_add, D)
            v (Tensor): 新しいバリューのテンソル、形状は (B, H, T_add, D)
        Returns:
            Tuple[Tensor, Tensor]: これまでの全てのキーとバリューのビュー、形状は (B, H, T_total, D)
        """
        logger.debug(f"KVキャッシュに挿入を実行 {layer_idx=} {k.shape=} {v.shape=} {self.pos=}")

        # 1) 初期化

        # KVキャッシュが初期化されていない場合
        if self.kv_cache is None:
            # KVキャッシュを初期化
            # データ型とデバイスが必要なため遅延初期化（lazy initialization）
            self.kv_cache = torch.empty(self.kv_shape, dtype=k.dtype, device=k.device)

        # Insert new keys/values to the cache and return the full cache so far

        # 現在の位置と追加するトークン数を取得
        B, H, T_add, D = k.size()

        # 書き込み開始位置t0と終了位置t1を計算 
        t0, t1 = self.pos, self.pos + T_add
        logger.debug(f"{t0=} {t1=}")

        # 書き込みの終了位置が初期化時のシーケンス長を超える場合、キャッシュを拡張
        if t1 > self.kv_cache.size(4):
            # 1024トークン分拡張
            t_needed = t1 + 1024

            # 1024の倍数に切り上げ
            t_needed = (t_needed + 1023) & ~1023

            additional_shape = list(self.kv_cache.shape)

            additional_shape[4] = t_needed - self.kv_cache.size(4)
            additional_cache = torch.empty(additional_shape, dtype=k.dtype, device=k.device)
            self.kv_cache = torch.cat([self.kv_cache, additional_cache], dim=4).contiguous()
            self.kv_shape = self.kv_cache.shape

        # 2) キーとバリューをキャッシュに挿入

        # キーをキャッシュに挿入
        self.kv_cache[layer_idx, 0, :, :, t0:t1] = k

        # バリューをキャッシュに挿入
        self.kv_cache[layer_idx, 1, :, :, t0:t1] = v

        # 3) これまでの全てのキーとバリューを返す

        # これまでの全てのキーを取得
        key_view = self.kv_cache[layer_idx, 0, :, :, :t1]

        # これまでの全てのバリューを取得
        value_view = self.kv_cache[layer_idx, 1, :, :, :t1]

        # 最後のTransformer層の場合、書き込み位置を更新
        if layer_idx == self.kv_cache.size(0) - 1:
            self.pos = t1

        logger.debug(f"KVキャッシュへの挿入完了 {key_view.shape=} {value_view.shape=} {self.pos=}")
        return key_view, value_view

In [ ]:
class RowState:
    """
    生成中の各行（サンプル）の状態を追跡するクラス
    """
    def __init__(self, current_tokens=None):

        # これまでに生成されたトークンIDの完全なリスト
        self.current_tokens = current_tokens or []

        # サンプリングしたトークンを無視し強制的に挿入するトークンのキュー
        # <|output_start|>などの特殊トークンの挿入に使用
        self.forced_tokens = deque()

        # <|python_start|>トークンを生成した時にTrueになるフラグ
        self.in_python_block = False

        # self.in_python_blockがTrueの間に生成されたトークンIDのリスト
        # Pythonのコードに対応するトークンID
        self.python_expr_tokens = []

        # 生成が完了したかどうかのフラグ
        # <|assistant_end|>や<|bos|>が生成された場合にTrueになる
        self.completed = False

In [ ]:
class Engine:
    """
    モデルの推論を実行するインターフェイス
    """

    def __init__(self, model, tokenizer):
        logger.debug(f"Engineを初期化開始 {model=} {tokenizer=}")
        
        self.model = model

        # ツール呼び出しで使用
        self.tokenizer = tokenizer

        logger.debug("Engineの初期化完了")

    @torch.inference_mode() # 勾配計算を無効化
    def generate(self, tokens, num_samples=1, max_tokens=None, temperature=1.0, top_k=None, seed=42):
        """
        トークンを1つずつストリーミングで生成するジェネレータ

        Args:
            tokens (List[int]): プロンプトのトークンIDのリスト
            num_samples (int): 生成するサンプル数（行数）
            max_tokens (int, optional): 生成する最大トークン数
            temperature (float): 温度パラメータ
            top_k (int, optional): Top-KサンプリングのK値
            seed (int): 乱数シード
        Yields:
            Tuple[List[int], List[int]]: 各行の次のトークンIDのリストと対応するマスクのリスト
        """
        logger.debug(f"ストリーミング生成を開始 {tokens=} {num_samples=} {max_tokens=} {temperature=} {top_k=} {seed=}")

        # 1) 初期化

        # 入力トークンの型を検証
        assert isinstance(tokens, list) and isinstance(tokens[0], int), "expecting list of ints"

        # デバイスを取得
        device = self.model.get_device()

        # 乱数生成器を初期化
        rng = torch.Generator(device=device)
        rng.manual_seed(seed)

        # 特殊トークンのIDを取得
        get_special = lambda s: self.tokenizer.encode_special(s)
        python_start = get_special("<|python_start|>")
        python_end = get_special("<|python_end|>")
        output_start = get_special("<|output_start|>")
        output_end = get_special("<|output_end|>")
        assistant_end = get_special("<|assistant_end|>") # if sampled, ends row
        bos = self.tokenizer.get_bos_token_id() # if sampled, ends row

        # 2) プロンプトのトークンを処理

        # KVキャッシュを初期化
        m = self.model.config
        kv_model_kwargs = {"num_heads": m.n_kv_head, "head_dim": m.n_embd // m.n_head, "num_layers": m.n_layer}
        kv_cache_prefill = KVCache(
            batch_size=1, # バッチサイズ1
            seq_len=len(tokens), # プロンプトの長さ
            **kv_model_kwargs, # KVキャッシュ設定
        )

        # トークンをテンソルに変換
        ids = torch.tensor([tokens], dtype=torch.long, device=device)
        logger.debug(f"{ids.shape=}")

        # モデルを順伝播してKVキャッシュを事前設定（prefill）
        logits = self.model.forward(ids, kv_cache=kv_cache_prefill)
        logger.debug(f"{logits.shape=}")

        # 最後のトークン（次の単語の予測）を抽出
        logits = logits[:, -1, :]
        logger.debug(f"{logits.shape=}")

        # 最初の1トークンをサンプリング    
        # (B, 1)
        next_ids = sample_next_token(logits, rng, temperature, top_k)
        logger.debug(f"{next_ids.shape=}")

        # サンプリングされたトークンをPythonのリストに変換
        sampled_tokens = next_ids[:, 0].tolist()
        logger.debug(f"{sampled_tokens=}")

        # 2) KVキャッシュの複製

        # 生成用のKVキャッシュを初期化
        kv_length_hint = (len(tokens) + max_tokens) if max_tokens is not None else self.model.config.sequence_len
        kv_cache_decode = KVCache(
            batch_size=num_samples,
            seq_len=kv_length_hint,
            **kv_model_kwargs,
        )
        logger.debug(f"{num_samples=} {kv_length_hint=}")

        # プロンプトで事前設定されたKVキャッシュを複製
        kv_cache_decode.prefill(kv_cache_prefill)

        # Prefill用のKVキャッシュは不要になったので削除
        del kv_cache_prefill

        # 3) 各サンプルの状態を初期化

        row_states = [RowState(tokens.copy()) for _ in range(num_samples)]
        logger.debug(f"{len(row_states)=}")

        # 4) 生成ループ

        # max_tokensに達するか、全サンプルが完了するまでトークンを生成
        num_generated = 0
        first_iteration = True
        while True:

            # 最大シーケンス長に達した場合、停止
            if max_tokens is not None and num_generated >= max_tokens:
                break

            # 全サンプルが完了した場合、停止
            if all(state.completed for state in row_states):
                break

            # 最初の生成ステップの場合
            if first_iteration:

                # プリフィルでサンプリングしたトークンを使用
                # すべてのサンプルにブロードキャスト
                sampled_tokens = [sampled_tokens[0]] * num_samples
                logger.debug(f"最初の生成ステップ {sampled_tokens=}")

                first_iteration = False

            # 2回目以降の生成ステップの場合
            else:

                # 順伝播を行い、各サンプルの次のトークンのロジットを計算
                # (B, T, vocab_size)
                logits = self.model.forward(ids, kv_cache=kv_cache_decode)
                logger.debug(f"2回目移行の生成ステップ {logits.shape=}")

                # 直近に生成されたトークンのロジットを抽出
                # (B, vocab_size)
                logits = logits[:, -1, :]
                logger.debug(f"{logits.shape=}")

                # 各サンプルの次のトークンをサンプリング
                # (B, 1)
                next_ids = sample_next_token(logits, rng, temperature, top_k)
                logger.debug(f"{next_ids.shape=}")

                # リストに変換
                sampled_tokens = next_ids[:, 0].tolist()
                logger.debug(f"{sampled_tokens=}")

            # それぞれのサンプルを処理
            token_column = [] # contains the next token id along each row
            token_masks = [] # contains the mask (was it sampled (1) or forced (0)?) along each row
            for i, state in enumerate(row_states):

                # 強制的に挿入するトークンがあるかを確認
                is_forced = len(state.forced_tokens) > 0

                # マスクを作成 
                # 0なら強制トークン、1ならサンプリングトークン
                token_masks.append(0 if is_forced else 1)

                # 強制トークンを優先して次のトークンを決定
                next_token = state.forced_tokens.popleft() if is_forced else sampled_tokens[i]

                # トークン列に次のトークンを追加
                token_column.append(next_token)

                # 状態の現在のトークン列に次のトークンを追加
                state.current_tokens.append(next_token)

                # 次のトークンが<|assistant_end|>または<|bos|>の場合
                if next_token == assistant_end or next_token == bos:
                    # 生成完了フラグを立てる
                    state.completed = True

                # <|python_start|>トークンが生成された場合
                if next_token == python_start:
                    # Pythonコードブロックに入るフラグを立てる
                    state.in_python_block = True
                    # Pythonコードトークンのリストを初期化
                    state.python_expr_tokens = []

                # <|python_end|>トークンが生成された場合
                elif next_token == python_end and state.in_python_block:
                    # Pythonコードブロックから出るフラグを下ろす
                    state.in_python_block = False

                    # ツール呼び出しを実行
                    if state.python_expr_tokens:
                        # トークン列をPythonコードの文字列にデコード
                        expr = self.tokenizer.decode(state.python_expr_tokens)

                        # ツールを呼び出す
                        result = use_calculator(expr)

                        # 結果がある場合
                        if result is not None:
                            # 結果をトークン化
                            result_tokens = self.tokenizer.encode(str(result))

                            # 特殊トークンで囲んで強制挿入キューに追加
                            # <|output_start|> result <|output_end|>
                            state.forced_tokens.append(output_start)
                            state.forced_tokens.extend(result_tokens)
                            state.forced_tokens.append(output_end)

                    # Pythonコードトークンのリストをクリア
                    state.python_expr_tokens = []

                # Pythonコードブロック内の場合
                elif state.in_python_block:
                    # 生成されたトークンをPythonコードトークンのリストに追加
                    state.python_expr_tokens.append(next_token)

            # 各行の次のトークン列とマスクをストリーミング出力
            yield token_column, token_masks

            # 生成トークン数を加算
            num_generated += 1

            # 次のループのためにトークンIDをテンソルに変換
            ids = torch.tensor(token_column, dtype=torch.long, device=device).unsqueeze(1)

    def generate_batch(self, tokens, num_samples=1, **kwargs):
        """
        非ストリーミング版

        Args:
            tokens (List[int]): プロンプトのトークンIDのリスト
            num_samples (int): 生成するサンプル数（行数）
            **kwargs: generateメソッドに渡す追加の引数
        Returns:
            Tuple[List[List[int]], List[List[int]]]: 各サンプルの生成されたトークン列と対応するマスクのリスト
        """
        logger.debug(f"バッチ生成を開始 {tokens=} {num_samples=} {kwargs=}")

        # 1) 初期化

        # ストップトークンのIDを取得
        assistant_end = self.tokenizer.encode_special("<|assistant_end|>")
        bos = self.tokenizer.get_bos_token_id()

        # 入力プロンプトを各サンプルにコピー
        results = [tokens.copy() for _ in range(num_samples)]

        # 各サンプルのマスクを初期化
        # 0は強制的に挿入されたトークン、1はサンプリングされたトークン
        masks = [[0] * len(tokens) for _ in range(num_samples)]

        # 各サンプルの生成完了フラグを初期化
        completed = [False] * num_samples

        # 2) 生成ループ

        # ストリーミングでトークンを生成
        for token_column, token_masks in self.generate(tokens, num_samples, **kwargs):

            # 各サンプルの生成されたトークンとマスクを収集
            for i, (token, mask) in enumerate(zip(token_column, token_masks)):

                # 未完了の場合
                if not completed[i]:

                    # 終了トークンの場合
                    if token == assistant_end or token == bos:

                        # 完了フラグを立てる
                        completed[i] = True

                    # 通常のトークンの場合
                    else:
                        # トークンを追加
                        results[i].append(token)

                        # マスクを追加
                        masks[i].append(mask)

            # すべての完了フラグが立っている場合、ループを終了
            if all(completed):
                break

        logger.debug(f"バッチ生成完了 {results=} {masks=}")
        return results, masks

In [ ]:
@torch.inference_mode()
def sample_next_token(logits, rng, temperature=1.0, top_k=None):
    """Sample a single next token from given logits of shape (B, vocab_size). Returns (B, 1)."""
    assert temperature >= 0.0, "temperature must be non-negative"
    if temperature == 0.0:
        return torch.argmax(logits, dim=-1, keepdim=True)
    if top_k is not None:
        k = min(top_k, logits.size(-1))
        vals, idx = torch.topk(logits, k, dim=-1)
        vals = vals / temperature
        probs = F.softmax(vals, dim=-1)
        choice = torch.multinomial(probs, num_samples=1, generator=rng)
        return idx.gather(1, choice)
    else:
        logits = logits / temperature
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, num_samples=1, generator=rng)

### モデルのロード

In [ ]:
import os
import re
import glob
import json
import logging
import torch

In [ ]:
def load_checkpoint(checkpoint_dir, step, device, load_optimizer=False):
    # Load the model state
    model_path = os.path.join(checkpoint_dir, f"model_{step:06d}.pt")
    model_data = torch.load(model_path, map_location=device)
    # Load the optimizer state if requested
    optimizer_data = None
    if load_optimizer:
        optimizer_path = os.path.join(checkpoint_dir, f"optim_{step:06d}.pt")
        optimizer_data = torch.load(optimizer_path, map_location=device)
    # Load the metadata
    meta_path = os.path.join(checkpoint_dir, f"meta_{step:06d}.json")
    with open(meta_path, "r") as f:
        meta_data = json.load(f)
    return model_data, optimizer_data, meta_data

In [ ]:
def build_model(checkpoint_dir, step, device, phase):
    """
    A bunch of repetitive code to build a model from a given checkpoint.
    Returns:
    - base model - uncompiled, not wrapped in DDP
    - tokenizer
    - meta data saved during base model training
    """
    assert phase in ["train", "eval"], f"Invalid phase: {phase}"
    model_data, optimizer_data, meta_data = load_checkpoint(checkpoint_dir, step, device, load_optimizer=False)
    # Hack: fix torch compile issue, which prepends all keys with _orig_mod.
    model_data = {k.lstrip("_orig_mod."): v for k, v in model_data.items()}
    model_config_kwargs = meta_data["model_config"]
    logger.debug(f"Building model with config: {model_config_kwargs}")
    model_config = GPTConfig(**model_config_kwargs)
    with torch.device("meta"):
        model = GPT(model_config)
    # Load the model state
    model.to_empty(device=device)
    model.init_weights() # note: this is dumb, but we need to init the rotary embeddings. TODO: fix model re-init
    model.load_state_dict(model_data, strict=True, assign=True)
    # Put the model in the right training phase / mode
    if phase == "eval":
        model.eval()
    else:
        model.train()
    # Load the Tokenizer
    tokenizer = get_tokenizer()
    # Sanity check: compatibility between model and tokenizer
    assert tokenizer.get_vocab_size() == model_config_kwargs["vocab_size"]
    return model, tokenizer, meta_data

In [ ]:
def load_model_from_dir(checkpoints_dir, device, phase, model_tag=None, step=None):
    if model_tag is None:
        # guess the model tag by defaulting to the largest model
        model_tag = find_largest_model(checkpoints_dir)
        logger.warn(f"No model tag provided, guessing model tag: {model_tag}")
    checkpoint_dir = os.path.join(checkpoints_dir, model_tag)
    if step is None:
        # guess the step by defaulting to the last step
        step = find_last_step(checkpoint_dir)
    assert step is not None, f"No checkpoints found in {checkpoint_dir}"
    # build the model
    logger.debug(f"Loading model from {checkpoint_dir} with step {step}")
    model, tokenizer, meta_data = build_model(checkpoint_dir, step, device, phase)
    return model, tokenizer, meta_data

In [ ]:
def load_model(source, *args, **kwargs):
    model_dir = {
        "base": "base_checkpoints",
        "mid": "mid_checkpoints",
        "sft": "chatsft_checkpoints",
        "rl": "chatrl_checkpoints",
    }[source]
    base_dir = get_base_dir()
    checkpoints_dir = os.path.join(base_dir, model_dir)
    return load_model_from_dir(checkpoints_dir, *args, **kwargs)

# Load the model and tokenizer
device = torch.device("cuda")
model_tag = "d20"
step = 21400
model, tokenizer, meta = load_model("base", device, phase="train", model_tag=model_tag, step=step)

## 中間学習

In [ ]:
# 合成データをダウンロード

import os

if not os.path.exists(f"{get_base_dir()}/identity_conversations.jsonl"):
    !curl -L -o {get_base_dir()}/identity_conversations.jsonl https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl

In [ ]:
from collections import deque
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import time
import wandb
import torch
from contextlib import nullcontext
# from nanochat.common import compute_init, compute_cleanup, logger.debug, DummyWandb, get_base_dir, autodetect_device_type
# from nanochat.tokenizer import get_token_bytes
# from nanochat.checkpoint_manager import save_checkpoint
# from nanochat.loss_eval import evaluate_bpb
# from nanochat.checkpoint_manager import load_model
import torch.distributed as dist

### ハイパーパラメータの設定

In [ ]:
# Number of processes/GPUs to use
NPROC_PER_NODE=1 # 8

In [ ]:
# -----------------------------------------------------------------------------
run = "dummy" # wandb run name default ("dummy" is special - we won't log to wandb)
device_type = "" # cuda|cpu|mps (empty => autodetect)
model_tag = None # model tag to load the model from (base model or midtrained model)
step = None # step to load the model from (base model or midtrained model)
dtype = "bfloat16"
# default -1
num_iterations = 1 # explicit number of steps of the optimization (-1 = disable)
max_seq_len = 2048
device_batch_size = 8 # 32
unembedding_lr = 0.004
embedding_lr = 0.2
matrix_lr = 0.02
init_lr_frac = 1.0 # initial learning rate is this fraction of the base learning rate
weight_decay = 0.0
eval_every = 150 # -1 = disable
eval_tokens = 20*524288
total_batch_size = 524288
dry_run = 0 # dry_run=1 is for experiments: we will log to wandb but we won't write checkpoints or report
config_keys = [k for k,v in globals().items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))]
# exec(open(os.path.join("nanochat", "nanochat", "configurator.py")).read()) # overrides from command line or config file
user_config = {k: globals()[k] for k in config_keys} # possibly useful for logging
user_config

### 分散処理の初期化

In [ ]:
def autodetect_device_type():
    # prefer to use CUDA if available, otherwise use MPS, otherwise fallback on CPU
    if torch.cuda.is_available():
        device_type = "cuda"
    elif torch.backends.mps.is_available():
        device_type = "mps"
    else:
        device_type = "cpu"
    logger.debug(f"Autodetected device type: {device_type}")
    return device_type

In [ ]:
def get_dist_info():
    if is_ddp():
        assert all(var in os.environ for var in ['RANK', 'LOCAL_RANK', 'WORLD_SIZE'])
        ddp_rank = int(os.environ['RANK'])
        ddp_local_rank = int(os.environ['LOCAL_RANK'])
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        return True, ddp_rank, ddp_local_rank, ddp_world_size
    else:
        return False, 0, 0, 1

In [ ]:
def is_ddp():
    # TODO is there a proper way
    return int(os.environ.get('RANK', -1)) != -1

In [ ]:
def compute_init(device_type="cuda"): # cuda|cpu|mps
    """Basic initialization that we keep doing over and over, so make common."""

    assert device_type in ["cuda", "mps", "cpu"], "Invalid device type atm"
    if device_type == "cuda":
        assert torch.cuda.is_available(), "Your PyTorch installation is not configured for CUDA but device_type is 'cuda'"
    if device_type == "mps":
        assert torch.backends.mps.is_available(), "Your PyTorch installation is not configured for MPS but device_type is 'mps'"

    # Reproducibility
    # Note that we set the global seeds here, but most of the code uses explicit rng objects.
    # The only place where global rng might be used is nn.Module initialization of the model weights.
    torch.manual_seed(42)
    if device_type == "cuda":
        torch.cuda.manual_seed(42)
    # skipping full reproducibility for now, possibly investigate slowdown later
    # torch.use_deterministic_algorithms(True)

    # Precision
    if device_type == "cuda":
        torch.set_float32_matmul_precision("high") # uses tf32 instead of fp32 for matmuls

    # Distributed setup: Distributed Data Parallel (DDP), optional, and requires CUDA
    ddp, ddp_rank, ddp_local_rank, ddp_world_size = get_dist_info()
    if ddp and device_type == "cuda":
        device = torch.device("cuda", ddp_local_rank)
        torch.cuda.set_device(device)  # make "cuda" default to this device
        dist.init_process_group(backend="nccl", device_id=device)
        dist.barrier()
    else:
        device = torch.device(device_type) # mps|cpu

    if ddp_rank == 0:
        logger.debug(f"Distributed world size: {ddp_world_size}")

    return ddp, ddp_rank, ddp_local_rank, ddp_world_size, device

In [ ]:
# Compute init
device_type = autodetect_device_type() if device_type == "" else device_type
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init(device_type)
master_process = ddp_rank == 0
autocast_ctx = torch.amp.autocast(device_type=device_type, dtype=torch.bfloat16) if device_type == "cuda" else nullcontext()
synchronize = torch.cuda.synchronize if device_type == "cuda" else lambda: None
get_max_memory = torch.cuda.max_memory_allocated if device_type == "cuda" else lambda: 0

### WandBの初期化

In [ ]:
# wandb logging init

class DummyWandb:
    """Useful if we wish to not use wandb but have all the same signatures"""
    def __init__(self):
        pass
    def log(self, *args, **kwargs):
        pass
    def finish(self):
        pass


use_dummy_wandb = run == "dummy" or not master_process
wandb_run = DummyWandb() if use_dummy_wandb else wandb.init(project="nanochat-mid", name=run, config=user_config)

In [ ]:
pretrain_batch_size = meta.get("device_batch_size", None)
if pretrain_batch_size is not None and device_batch_size > pretrain_batch_size:
    logger.debug(f"FOOTGUN WARNING: base model training used device_batch_size {pretrain_batch_size}, did you pass in a good --device_batch_size to this script?")

pretrain_batch_size

In [ ]:
orig_model = model
model = torch.compile(model, dynamic=False)
depth = model.config.n_layer

In [ ]:
num_flops_per_token = model.estimate_flops()

In [ ]:
tokens_per_fwdbwd = device_batch_size * max_seq_len # tokens per iteration for a single rank
world_tokens_per_fwdbwd = tokens_per_fwdbwd * ddp_world_size # total tokens per iteration for all ranks
assert total_batch_size % world_tokens_per_fwdbwd == 0
grad_accum_steps = total_batch_size // world_tokens_per_fwdbwd
logger.debug(f"Tokens / micro-batch / rank: {device_batch_size} x {max_seq_len} = {tokens_per_fwdbwd:,}")
logger.debug(f"Tokens / micro-batch: {world_tokens_per_fwdbwd:,}")
logger.debug(f"Total batch size {total_batch_size:,} => gradient accumulation steps: {grad_accum_steps}")

In [ ]:
def get_token_bytes(device="cpu"):
    base_dir = get_base_dir()
    tokenizer_dir = os.path.join(base_dir, "tokenizer")
    token_bytes_path = os.path.join(tokenizer_dir, "token_bytes.pt")
    assert os.path.exists(token_bytes_path), f"Token bytes not found at {token_bytes_path}? It gets written by tok_train.py"
    with open(token_bytes_path, "rb") as f:
        token_bytes = torch.load(f, map_location=device)
    return token_bytes

token_bytes = get_token_bytes(device=device)

### 最適化関数

In [ ]:
import torch
from torch import Tensor
import torch.distributed as dist

In [ ]:
@torch.compile # PyTorch2.0のJITコンパイラを使用
def zeropower_via_newtonschulz5(G: Tensor, steps: int) -> Tensor:
    """
    ニュートン・シュルツ反復法で勾配の直行化を近似する

    行列Gの直交行列は、Gの特異値分解G = USV^Tに対してUV^Tで計算できる
    特異値分解（SVD）は計算コストが高いため近似手法を用いる
    原点における傾きを最大化するように選択された係数を持つ5次の反復法を採用

    Args: 
        G (Tensor): 直行化する行列、形状は(..., m, n)
        steps (int): 反復回数
    Returns:
        Tensor: 直交化された行列、形状は(..., m, n)
    """

    assert G.ndim >= 2

    # 5次反復法のための係数
    a, b, c = (3.4445, -4.7750,  2.0315)

    # 勾配をbfloat16にダウンキャスト
    X = G.bfloat16()

    # 安定化のため行列を横長にする
    if G.size(-2) > G.size(-1):
        X = X.mT

    # 行列のスペクトルノルム（最大の特異値）を1以下に正規化
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)

    # ニュートン・シュワルツ反復法を適用
    for _ in range(steps):
        # A = X X^T
        A = X @ X.mT

        # B = b(X X^T) + c(X X^T)^2
        B = b * A + c * A @ A

        # X_k+1 = a X_k + (B) X_k 
        X = a * X + B @ X

    # 元の形状に戻す
    if G.size(-2) > G.size(-1):
        X = X.mT

    # 直交行列を返す
    return X

In [ ]:
class Muon(torch.optim.Optimizer):
    """
    Muon最適化関数 https://kellerjordan.github.io/posts/muon/

    内部的に標準的なSGDモーメンタムを実行し、その後に直交化の後処理ステップを実行する
    直行化ステップでは、各2Dパラメータの更新が最も近い直交行列に置き換えられる
    各更新を効率的に直交化するために、ニュートン・シュルツ反復を使用する
    これにより、GPU上でbfloat16で安定して実行できる利点がある

    注意:
    - この最適化関数は、埋め込み層、最終全結合層、0次元・1次元パラメータには使用しない（AdamWなどを使用）
    - 4Dの畳み込みフィルタに使用する場合、最後の3つの次元をフラット化すると機能する

    Args:
        lr (float): 内部SGDで使用される学習率
        momentum (float): 内部SGDで使用されるモーメンタム
        nesterov (bool): 内部SGDでNesterovスタイルのモーメンタムを使用するかどうか（推奨）
        ns_steps (int): ニュートン・シュルツ反復のステップ数
    """

    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        """
        Muon最適化関数の初期化

        Args:
            params (iterable): 最適化するパラメータ
            lr (float, optional): 内部SGDで使用される学習率
            momentum (float, optional): 内部SGDで使用されるモーメンタム
            nesterov (bool, optional): 内部SGDでNesterovスタイルのモーメンタムを使用するかどうか
            ns_steps (int, optional): ニュートン・シュルツ反復のステップ数
        """
        # 最適化関数のデフォルト設定
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)

        params: list[Tensor] = [*params]

        param_groups = []

        # 同じ要素数のパラメータをグループ化
        for size in {p.numel() for p in params}:
            group = dict(params=[p for p in params if p.numel() == size])
            param_groups.append(group)

        # 親クラスの初期化
        super().__init__(param_groups, defaults)

    @torch.no_grad()
    def step(self):
        """
        Muon最適化関数の1ステップの更新を実行
        """

        # 各パラメータグループでループ
        for group in self.param_groups:

            params: list[Tensor] = group["params"]

            # 各パラメータでループ
            for p in params:
                # 勾配を取得
                g = p.grad
                assert g is not None

                # モメンタムバッファを取得または初期化
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf: Tensor = state["momentum_buffer"]

                # モメンタムバッファを最新の勾配で更新（指数移動平均）
                buf.lerp_(g, 1 - group["momentum"])

                # ネステロフ・モーメンタムを使用する場合、勾配を調整
                g = g.lerp_(buf, group["momentum"]) if group["nesterov"] else buf

                # 勾配を直交化
                g = zeropower_via_newtonschulz5(g, steps=group["ns_steps"])

                # パラメータを更新
                # -group["lr"]: 学習率を負にして減少方向に更新
                # max(1, p.size(-2) / p.size(-1))**0.5: 行列の形状に基づいてスケーリング
                p.add_(g, alpha=-group["lr"] * max(1, p.size(-2) / p.size(-1))**0.5)

In [ ]:

# Initialize the Optimizer (Muon for Linear layers, AdamW for embedding and lm_head)
optimizers = model.setup_optimizers(unembedding_lr=unembedding_lr, embedding_lr=embedding_lr, matrix_lr=matrix_lr, weight_decay=weight_decay)
adamw_optimizer, muon_optimizer = optimizers
# Override the initial learning rate as a fraction of the base learning rate
for opt in optimizers:
    for group in opt.param_groups:
        group["lr"] = group["lr"] * init_lr_frac
        group["initial_lr"] = group["lr"] # save the initial learning so we can decay easily later

### データローダー

In [ ]:

import random

class Task:
    """
    Base class of a Task. Allows for lightweight slicing of the underlying dataset.
    """

    def __init__(self, start=0, stop=None, step=1):
        # allows a lightweight logical view over a dataset
        assert start >= 0, f"Start must be non-negative, got {start}"
        assert stop is None or stop >= start, f"Stop should be greater than or equal to start, got {stop} and {start}"
        assert step >= 1, f"Step must be strictly positive, got {step}"
        self.start = start
        self.stop = stop # could be None here
        self.step = step

    @property
    def eval_type(self):
        # one of 'generative' | 'categorical'
        raise NotImplementedError

    def num_examples(self):
        raise NotImplementedError

    def get_example(self, index):
        raise NotImplementedError

    def __len__(self):
        start = self.start
        stop = self.num_examples() if self.stop is None else self.stop
        step = self.step
        span = stop - start
        num = (span + step - 1) // step # ceil_div(span, step)
        assert num >= 0, f"Negative number of examples???: {num}" # prevent footguns
        return num

    def __getitem__(self, index: int):
        assert isinstance(index, int), f"Index must be an integer, got {type(index)}"
        physical_index = self.start + index * self.step
        conversation = self.get_example(physical_index)
        return conversation

    def evaluate(self, problem, completion):
        raise NotImplementedError


In [ ]:
"""
SmolTalk by HuggingFace. Good "general" conversational dataset.
https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk
We use the "smol" version, which is more appropriate for smaller models.
"""

from datasets import load_dataset

class SmolTalk(Task):
    """ smol-smoltalk dataset. train is 460K rows, test is 24K rows. """

    def __init__(self, split, **kwargs):
        super().__init__(**kwargs)
        assert split in ["train", "test"], "SmolTalk split must be train|test"
        self.ds = load_dataset("HuggingFaceTB/smol-smoltalk", split=split).shuffle(seed=42)
        self.length = len(self.ds)

    def num_examples(self):
        return self.length

    def get_example(self, index):
        row = self.ds[index]
        messages = row["messages"]
        # ---------------------------------------------------------------------
        # sanity checking asserts here
        # TODO: we could remove these asserts later, for now just don't want any footguns
        # there is an optional system message at the beginning
        assert len(messages) >= 1
        first_message = messages[0]
        if first_message["role"] == "system":
            rest_messages = messages[1:] # optional system message is OK
        else:
            rest_messages = messages
        assert len(rest_messages) >= 2, "SmolTalk messages must have at least 2 messages"
        for i, message in enumerate(rest_messages):
            # user and assistant alternate as user,assistant,user,assistant,...
            expected_role = "user" if i % 2 == 0 else "assistant"
            assert message["role"] == expected_role, f"Message {i} has role {message['role']} but should be {expected_role}"
            assert isinstance(message["content"], str), "Content must be a string"
        # ---------------------------------------------------------------------
        # create and return the Conversation object (ok to emit the system message too)
        conversation = {
            "messages": messages,
        }
        return conversation

In [ ]:

class MMLU(Task):

    letters = ('A', 'B', 'C', 'D')
    groups = ('abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management', 'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios', 'nutrition', 'philosophy', 'prehistory', 'professional_accounting', 'professional_law', 'professional_medicine', 'professional_psychology', 'public_relations', 'security_studies', 'sociology', 'us_foreign_policy', 'virology', 'world_religions')

    def __init__(self, subset, split, **kwargs):
        super().__init__(**kwargs)
        assert subset in ["all", "auxiliary_train"], f"subset {subset} must be all|auxiliary_train"
        assert split in ["train", "validation", "dev", "test"], f"split {split} must be train|validation|dev|test"
        if subset == "auxiliary_train":
            assert split == "train", "auxiliary_train must be split into train"
        self.subset = subset
        self.split = split
        self.ds = load_dataset("cais/mmlu", subset, split=split).shuffle(seed=42)
        if subset == "auxiliary_train":
            # I don't understand why but the auxiliary_train rows have some weird additional 'train' wrapper
            self.ds = self.ds.map(lambda row: row['train'], remove_columns=['train'])

    @property
    def eval_type(self):
        return 'categorical'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        row = self.ds[index]
        question = row["question"] # the question text
        choices = row["choices"] # the text of each choice
        answer = row["answer"] # index of the answer, e.g. 0,1,2,3 (for A,B,C,D)
        subject = row["subject"] # e.g. "college_biology", "college_chemistry", etc.
        assert len(choices) == 4, "MMLU should have 4 choices"
        # create and return the Conversation object
        user_message = render_mc(question, self.letters, choices)
        assistant_message = self.letters[answer]
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]
        conversation = {
            "messages": messages,
            "subject": subject, # might be useful later for grouping metrics by subject
            "letters": self.letters, # useful during evaluation, so we can narrow and clamp the assistant prediction to one of the letters
        }
        return conversation

    def evaluate(self, conversation, assistant_response):
        # the assert here is not strictly speaking needed, but currently the way we eval, we expect this to be true
        # I'm going to leave the assert here to prevent footguns, but possibly in the future can remove it.
        assert assistant_response in self.letters, f"MMLU answer {assistant_response} is expected to be one of {self.letters}"
        assistant_message = conversation['messages'][-1]['content'] # e.g. "A"
        return assistant_response == assistant_message

In [ ]:
GSM_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
def extract_answer(completion):
    """
    Extract the numerical answer after #### marker.
    Follows official code for normalization:
    https://github.com/openai/grade-school-math/blob/3101c7d5072418e28b9008a6636bde82a006892c/grade_school_math/dataset.py#L28
    """
    match = GSM_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    return None


class GSM8K(Task):

    def __init__(self, subset, split, **kwargs):
        super().__init__(**kwargs)
        assert subset in ["main", "socratic"], "GSM8K subset must be main|socratic"
        assert split in ["train", "test"], "GSM8K split must be train|test"
        self.ds = load_dataset("openai/gsm8k", subset, split=split).shuffle(seed=42)

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        """ Get a single problem from the dataset. """
        row = self.ds[index]
        question = row['question'] # string of the question prompt
        answer = row['answer'] # string of the full solution and the answer after #### marker
        # Create and return the Conversation object
        # This is tricky because GSM8K uses tool calls, which we need to parse here.
        assistant_message_parts = []
        parts = re.split(r'(<<[^>]+>>)', answer)
        for part in parts:
            if part.startswith('<<') and part.endswith('>>'):
                # This is a calculator tool call
                inner = part[2:-2]  # Remove << >>
                # Split on = to get expression and result
                if '=' in inner:
                    expr, result = inner.rsplit('=', 1)
                else:
                    expr, result = inner, ""
                # Add the tool call as a part
                assistant_message_parts.append({"type": "python", "text": expr})
                # Add the result as a part
                assistant_message_parts.append({"type": "python_output", "text": result})
            else:
                # Regular text in between tool calls
                assistant_message_parts.append({"type": "text", "text": part})
        # No put it all together
        messages = [
            {"role": "user", "content": question}, # note: simple string
            {"role": "assistant", "content": assistant_message_parts}, # note: list of parts (as dicts)
        ]
        conversation = {
            "messages": messages,
        }
        return conversation

    def evaluate(self, conversation, assistant_response):
        """
        Given (conversation, completion), return evaluation outcome (0 = wrong, 1 = correct)
        Note that:
        - the conversation has both user AND assistant message (containing the ground truth answer)
        - the assistant_response is usually the alternative assistant message achieved via sampling

        TODO: Technically, assistant_response should be a Message (either a string or a list of parts)
              We can handle this later possibly. For now just assume string.
        """
        assert isinstance(assistant_response, str), "Assuming simple string response for now"
        # First extract the ground truth answer
        assistant_message = conversation['messages'][-1]
        assert assistant_message['role'] == "assistant", "Last message must be from the Assistant"
        assert isinstance(assistant_message['content'], list), "This is expected to be a list of parts"
        last_text_part = assistant_message['content'][-1]['text'] # this contains the final answer in GSM8K
        # Extract both the ground truth answer and the predicted answer
        ref_num = extract_answer(last_text_part)
        pred_num = extract_answer(assistant_response)
        # Compare and return the success as int
        is_correct = int(pred_num == ref_num)
        return is_correct

    def reward(self, conversation, assistant_response):
        """
        Used during RL. To keep things simple, just re-use the evaluation above.
        Later this could be made more complex (e.g. format matching etc.)
        """
        is_correct = self.evaluate(conversation, assistant_response)
        is_correct_float = float(is_correct)
        return is_correct_float


In [ ]:

class CustomJSON(Task):
    """
    Load conversations from a JSONL file.
    Each line should be a JSON array of message objects with 'role' and 'content' fields.
    Example line: [{"role":"user","content":"Hi"},{"role":"assistant","content":"Hello"}]
    """

    def __init__(self, filepath, **kwargs):
        super().__init__(**kwargs)
        self.filepath = filepath
        self.conversations = []

        # Load all conversations from the JSONL file
        if not os.path.exists(filepath):
            # Helpful error message due to recent change. Will be removed in the future.
            print("-" * 80)
            print(f"Warning: File {filepath} does not exist")
            print("HINT (Oct 21 2025)")
            print("If you recently did a git pull and suddely see this, it might be due to the new addition of identity conversations")
            print("See this discussion for more details: https://github.com/karpathy/nanochat/discussions/139")
            print("Quick fix: simply run the following command to download the file and you're done:")
            print(f"curl -L -o {filepath} https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl")
            print("-" * 80)

        else:
            with open(filepath, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:  # skip empty lines
                        continue
                    messages = json.loads(line)
                    # Validate the conversation structure
                    assert isinstance(messages, list), f"Expected list of messages, got {type(messages)}"
                    assert len(messages) >= 2, f"Conversation must have at least 2 messages, got {len(messages)}"
                    # Validate message structure and alternating roles
                    for i, message in enumerate(messages):
                        assert "role" in message, f"Message {i} missing 'role' field"
                        assert "content" in message, f"Message {i} missing 'content' field"
                        expected_role = "user" if i % 2 == 0 else "assistant"
                        assert message["role"] == expected_role, f"Message {i} has role {message['role']} but should be {expected_role}"
                        assert isinstance(message["content"], str), f"Message {i} content must be a string"

                    self.conversations.append(messages)

        self.length = len(self.conversations)

    def num_examples(self):
        return self.length

    def get_example(self, index):
        messages = self.conversations[index]
        conversation = {
            "messages": messages,
        }
        return conversation

In [ ]:
WORD_LIST_URL = "https://raw.githubusercontent.com/dwyl/english-words/refs/heads/master/words_alpha.txt"

class SimpleSpelling(Task):
    """Much simpler task designed to get the model to just practice spelling words."""

    def __init__(self, size=1000, split="train", **kwargs):
        super().__init__(**kwargs)
        assert split in ["train", "test"], "SpellingBee split must be train|test"
        self.size = size
        self.split = split
        filename = WORD_LIST_URL.split("/")[-1]
        word_list_path = download_file_with_lock(WORD_LIST_URL, filename)
        with open(word_list_path) as f:
            words = [line.strip() for line in f]
        rng = random.Random(42)
        rng.shuffle(words) # use a different word order than the SpellingBee task
        self.words = words

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return self.size

    def get_example(self, index):
        seed = index if self.split == "train" else -(index + 1) # avoid collision at 0
        rng = random.Random(seed)
        # pick a random word
        word = rng.choice(self.words)
        word_letters = ",".join(list(word))
        # return the full conversation
        messages = [
            {"role": "user", "content": f"Spell the word: {word}"},
            {"role": "assistant", "content": f"{word}:{word_letters}"}
        ]
        conversation = {
            "messages": messages,
        }
        return conversation

In [ ]:

# Letters of the alphabet
LETTERS = "abcdefghijklmnopqrstuvwxyz"
# A list of 370K English words of large variety
WORD_LIST_URL = "https://raw.githubusercontent.com/dwyl/english-words/refs/heads/master/words_alpha.txt"

# Identical to gsm8k's answer extraction
ANSWER_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
def extract_answer(completion):
    """
    Extract the numerical answer after #### marker.
    """
    match = ANSWER_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    return None

# User message templates for data augmentation
USER_MSG_TEMPLATES = [
    "How many {letter} are in the word {word}",
    "How many {letter} are in {word}",
    "Count the number of {letter} in {word}",
    "How many times does {letter} appear in {word}",
    "What's the count of {letter} in {word}",
    "In the word {word}, how many {letter} are there",
    "How many letter {letter} are in the word {word}",
    "Count how many {letter} appear in {word}",
    "Tell me the number of {letter} in {word}",
    "How many occurrences of {letter} are in {word}",
    "Find the count of {letter} in {word}",
    "Can you count the {letter} letters in {word}",
    "What is the frequency of {letter} in {word}",
    "How many {letter}s are in {word}",
    "How many {letter}'s are in {word}",
    "Count all the {letter} in {word}",
    "How many times is {letter} in {word}",
    "Number of {letter} in {word}",
    "Total count of {letter} in {word}",
    "How many {letter} does {word} have",
    "How many {letter} does {word} contain",
    "What's the number of {letter} in {word}",
    "{word} has how many {letter}",
    "In {word}, count the {letter}",
    "How many {letter} appear in {word}",
    "Count the {letter} in {word}",
    "Give me the count of {letter} in {word}",
    "How many instances of {letter} in {word}",
    "Show me how many {letter} are in {word}",
    "Calculate the number of {letter} in {word}",
    # Spanish
    "¿Cuántas {letter} hay en {word}?",
    "¿Cuántas veces aparece {letter} en {word}?",
    "Cuenta las {letter} en {word}",
    "¿Cuántas letras {letter} tiene {word}?",
    # Chinese (Simplified)
    "{word}中有多少个{letter}",
    "{word}里有几个{letter}",
    "数一下{word}中的{letter}",
    "{word}这个词里有多少{letter}",
    # Korean
    "{word}에 {letter}가 몇 개 있나요",
    "{word}에서 {letter}의 개수는",
    "{word}에 {letter}가 몇 번 나오나요",
    "{word}라는 단어에 {letter}가 몇 개",
    # French
    "Combien de {letter} dans {word}",
    "Combien de fois {letter} apparaît dans {word}",
    "Compte les {letter} dans {word}",
    # German
    "Wie viele {letter} sind in {word}",
    "Wie oft kommt {letter} in {word} vor",
    "Zähle die {letter} in {word}",
    # Japanese
    "{word}に{letter}は何個ありますか",
    "{word}の中に{letter}がいくつ",
    "{word}に{letter}が何回出てくる",
]

class SpellingBee(Task):

    def __init__(self, size=1000, split="train", **kwargs):
        super().__init__(**kwargs)
        assert split in ["train", "test"], "SpellingBee split must be train|test"
        self.size = size
        self.split = split
        filename = WORD_LIST_URL.split("/")[-1]
        word_list_path = download_file_with_lock(WORD_LIST_URL, filename)
        with open(word_list_path) as f:
            words = [line.strip() for line in f]
        self.words = words

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return self.size

    def get_example(self, index):
        seed = index if self.split == "train" else -(index + 1) # avoid collision at 0
        rng = random.Random(seed)

        # pick a random word
        word = rng.choice(self.words)
        # pick a letter from it (90%) or a random letter (10%)
        letter = rng.choice(word) if rng.random() < 0.9 else rng.choice(LETTERS)

        # get the correct answer by simply counting
        count = word.count(letter)

        # create a user message, with a bunch of variations as data augmentation
        template = rng.choice(USER_MSG_TEMPLATES)
        # 30% chance to lowercase the template (lazy people don't use shift)
        if rng.random() < 0.3:
            template = template.lower()
        quote_options = ['', "'", '"']
        letter_quote = rng.choice(quote_options) # is the letter quoted?
        word_quote = rng.choice(quote_options) # is the word quoted?
        letter_wrapped = f"{letter_quote}{letter}{letter_quote}"
        word_wrapped = f"{word_quote}{word}{word_quote}"
        user_msg = template.format(letter=letter_wrapped, word=word_wrapped)
        if rng.random() < 0.5: # 50% of people don't even use question marks
            user_msg += "?"

        # Now create the ideal assistant response - build as parts (text + tool calls)
        assistant_parts = []
        word_letters = ",".join(list(word))
        manual_text = f"""We are asked to find the number '{letter}' in the word '{word}'. Let me try a manual approach first.

First spell the word out:
{word}:{word_letters}

Then count the occurrences of '{letter}':
"""
        # Little simulated loop of the solution process
        # TODO: This is where the fun starts, we could simulate cute little mistakes
        # and get the model to review its work and recover from them.
        # You might of course hope this could arise in RL too, but realistically you'd want to help it out a bit.
        running_count = 0
        for i, char in enumerate(word, 1):
            if char == letter:
                running_count += 1
                # note: there deliberately cannot be a space here between i and char
                # because this would create a different token! (e.g. " a" and "a" are different tokens)
                manual_text += f"{i}:{char} hit! count={running_count}\n"
            else:
                manual_text += f"{i}:{char}\n"

        manual_text += f"\nThis gives us {running_count}."
        assistant_parts.append({"type": "text", "text": manual_text})
        # Part 2: Python verification
        assistant_parts.append({"type": "text", "text": "\n\nLet me double check this using Python:\n\n"})
        # Part 3: Python tool call
        python_expr = f"'{word}'.count('{letter}')"
        assistant_parts.append({"type": "python", "text": python_expr})
        # Part 4: Python output
        assistant_parts.append({"type": "python_output", "text": str(count)})
        # Part 5: Final answer
        assistant_parts.append({"type": "text", "text": f"\n\nPython gives us {count}.\n\nMy final answer is:\n\n#### {count}"})

        # return the full conversation
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_parts}
        ]
        conversation = {
            "messages": messages,
        }
        return conversation

    def evaluate(self, conversation, assistant_response):
        """
        Given (conversation, completion), return evaluation outcome (0 = wrong, 1 = correct)
        Identical to gsm8k's evaluation.
        """
        assert isinstance(assistant_response, str), "Assuming simple string response for now"
        # First extract the ground truth answer from the conversation
        assistant_message = conversation['messages'][-1]
        assert assistant_message['role'] == "assistant", "Last message must be from the Assistant"
        assert isinstance(assistant_message['content'], list), "This is expected to be a list of parts"
        # The last text part contains the final answer with ####
        last_text_part = assistant_message['content'][-1]['text']
        # Extract both the ground truth answer and the predicted answer
        ref_num = extract_answer(last_text_part)
        pred_num = extract_answer(assistant_response)
        # Compare and return the success as int
        is_correct = int(pred_num == ref_num)
        return is_correct

    def reward(self, conversation, assistant_response):
        """ Use simple 0-1 reward just like gsm8k."""
        is_correct = self.evaluate(conversation, assistant_response)
        is_correct_float = float(is_correct)
        return is_correct_float


In [ ]:
class TaskMixture(Task):
    """
    For SFT Training it becomes useful to train on a tax mixture of datasets.
    Fun trick: if you wish to oversample any task, just pass it in multiple times in the list.
    """

    def __init__(self, tasks, **kwargs):
        super().__init__(**kwargs)
        # tasks is a list of Task objects
        self.tasks = tasks
        self.lengths = [len(task) for task in self.tasks]
        self.num_conversations = sum(self.lengths)
        # Build list of all (task_idx, local_idx) pairs
        self.index_map = []
        for task_idx, task_length in enumerate(self.lengths):
            for local_idx in range(task_length):
                self.index_map.append((task_idx, local_idx))
        # Deterministically shuffle to mix tasks throughout training
        rng = random.Random(42)
        rng.shuffle(self.index_map)
        # Note: this is not the most elegant or best solution, but it's ok for now

    def num_examples(self):
        return self.num_conversations

    def get_example(self, index):
        """
        Access conversations according to a deterministic shuffle of all examples.
        This ensures tasks are mixed throughout training, regardless of dataset size.
        """
        assert 0 <= index < self.num_conversations, f"Index {index} out of range for mixture with {self.num_conversations} conversations"
        task_idx, local_idx = self.index_map[index]
        return self.tasks[task_idx][local_idx]

In [ ]:
# Midtraining data mixture and DataLoader
base_dir = get_base_dir()
identity_conversations_filepath = os.path.join(base_dir, "identity_conversations.jsonl")
train_dataset = TaskMixture([
    SmolTalk(split="train"), # 460K rows of general conversations
    MMLU(subset="auxiliary_train", split="train"), # 100K rows of multiple choice problems drawn from ARC, MC_TEST, OBQA, RACE
    GSM8K(subset="main", split="train"), # 8K rows teaching simple math and (calculator) tool use
    CustomJSON(filepath=identity_conversations_filepath), # 1000 rows of synthetic identity conversations
    CustomJSON(filepath=identity_conversations_filepath), # let's do 2 epochs of these
    SimpleSpelling(size=200000, split="train"), # 200K rows of Simple Spelling (e.g. spell the word 'apple')
    SpellingBee(size=80000, split="train"), # 80K rows of Spelling Bee (e.g. how many 'r' are in 'strawberry'?)
]) # total: 460K + 100K + 8K + 200K + 80K = 848K rows

In [ ]:
val_dataset = TaskMixture([
    SmolTalk(split="test"), # 24K rows in test set
    MMLU(subset="all", split="test", stop=5200), # 14K rows in test set, use only 5.2K to match the train ratios
    GSM8K(subset="main", split="test", stop=420), # 1.32K rows in test set, use only 420 to match the train ratios
]) # total: 24K + 14K + 1.32K ~= 39K rows
# DataLoader is defined here, it emits inputs, targets : 2D tensors of shape (device_batch_size, max_seq_len)
# A big problem is that we don't know the final num_iterations in advance. So we create
# these two global variables and update them from within the data generator.

### 訓練

In [ ]:
# A big problem is that we don't know the final num_iterations in advance. So we create
# these two global variables and update them from within the data generator.
last_step = False # we will toggle this to True when we reach the end of the dataset
approx_progress = 0.0 # will go from 0 to 1 over the course of the epoch

def mid_data_generator(split):
    global last_step, approx_progress
    assert split in {"train", "val"}, "split must be 'train' or 'val'"
    dataset = train_dataset if split == "train" else val_dataset
    dataset_size = len(dataset)
    assert dataset_size > 0
    needed_tokens = device_batch_size * max_seq_len + 1 # to form one training batch of inputs,targets
    token_buffer = deque()
    # CUDA supports memory pinning for faster transfers between CPU and GPU:
    scratch = torch.empty(needed_tokens, dtype=torch.int64, pin_memory=(device_type == "cuda"))
    cursor = ddp_rank # increments by ddp_world_size each time, so each rank processes unique documents
    it = 0 # iteration counter
    while True:
        # Accumulate enough tokens for one iteration before yielding
        while len(token_buffer) < needed_tokens:
            conversation = dataset[cursor]
            ids, _ = tokenizer.render_conversation(conversation)
            token_buffer.extend(ids)
            cursor += ddp_world_size
            if cursor >= dataset_size:
                cursor -= dataset_size # wrap around for another epoch
                if split == "train":
                    last_step = True # toggle last_step to True, which will terminate the training loop
        # Stopping condition to respect num_iterations, if given
        it += 1
        if num_iterations > 0 and it >= num_iterations:
            last_step = True # toggle last_step to True, which will terminate the training loop
        # Build up inputs/targets and yield
        for i in range(needed_tokens):
            scratch[i] = token_buffer.popleft()
        inputs_cpu = scratch[:-1].to(dtype=torch.int32)
        targets_cpu = scratch[1:]
        inputs = inputs_cpu.view(device_batch_size, max_seq_len).to(device=device, dtype=torch.int32, non_blocking=True)
        targets = targets_cpu.view(device_batch_size, max_seq_len).to(device=device, dtype=torch.int64, non_blocking=True)
        if split == "train":
            if num_iterations > 0:
                approx_progress = it / num_iterations # calculate progress from the max number of iterations
            else:
                approx_progress = cursor / dataset_size # approximate progress as a fraction of the dataset
        yield inputs, targets

In [ ]:
train_loader = mid_data_generator("train")
build_val_loader = lambda: mid_data_generator("val")
progress = 0 # will go from 0 to 1 over the course of the epoch

In [ ]:
# Learning rate scheduler
def get_lr_multiplier(progress):
    # first 80% of training: no decay, then linearly ramp down to 0.
    return 1 if progress < 0.8 else 1 - (progress - 0.8) / 0.2

# Momentum scheduler for Muon optimizer
def get_muon_momentum(it):
    frac = min(it / 300, 1)
    momentum = (1 - frac) * 0.85 + frac * 0.95
    return momentum

In [ ]:
def render_mc(question, letters, choices):
    """
    The common multiple choice rendering format we will use.

    Note two important design decisions:
    1)
    Bigger models don't care as much, but smaller models prefer to have
    the letter *after* the choice, which results in better binding.
    2)
    There is no whitespace between the delimiter (=) and the letter.
    This is actually critical because the tokenizer has different token ids
    for " A" vs. "A". The assistant responses will be just the letter itself,
    i.e. "A", so it is important that here in the prompt it is the exact same
    token, i.e. "A" with no whitespace before it. Again, bigger models don't care
    about this too much, but smaller models do care about some of these details.
    """
    query = f"Multiple Choice question: {question}\n"
    query += "".join([f"- {choice}={letter}\n" for letter, choice in zip(letters, choices)])
    query += "\nRespond only with the letter of the correct answer."
    return query

In [ ]:
@torch.no_grad()
def evaluate_bpb(model, batches, steps, token_bytes):
    """
    BPB (bits per byte)でモデルを評価
    合計損失と合計バイト数を計算し、それらを割り算して正規化する
    """

    # 1) 初期化

    # 損失を記録するテンソルを初期化
    total_nats = torch.tensor(0.0, dtype=torch.float32, device=model.get_device())

    # モデルが予測したトークンの合計バイト数を記録するテンソルを初期化
    total_bytes = torch.tensor(0, dtype=torch.int64, device=model.get_device())

    # 検証データローダーのイテレーターを作成
    batch_iter = iter(batches)

    # 2) 評価ループ

    # 指定されたステップ数だけ評価を実行
    for _ in range(steps):

        # 入力シーケンスと正解シーケンスのバッチを取得
        x, y = next(batch_iter)

        # モデルの順伝搬を実行し、各トークンの損失を計算
        # loss_reduction='none'により、各トークンの損失が返される
        # (B, T)
        loss2d = model(x, y, loss_reduction='none')

        # フラット化
        # (B*T,)
        loss2d = loss2d.view(-1)

        # 正解トークンをフラット化
        # (B*T,)
        y = y.view(-1)

        # 正解シーケンスにignore_index（-1）が含まれている場合、それを除いて集計
        if (y.int() < 0).any():

            # マスクを作成
            valid = y >= 0

            # 無効なインデックスを0に置き換え
            y_safe = torch.where(valid, y, torch.zeros_like(y))

            # マスクされていないトークンをバイト数に変換
            num_bytes2d = torch.where(
                valid,
                token_bytes[y_safe],
                torch.zeros_like(y, dtype=token_bytes.dtype)
            )

            # ignore_indexと特殊トークンを除外して（num_bytes2d > 0）、損失を集計
            total_nats += (loss2d * (num_bytes2d > 0)).sum()

            # バイト数を集計
            total_bytes += num_bytes2d.sum()

        # 正解シーケンスにignore_indexが含まれていない場合、普通に集計
        else:
            # トークンをバイト数に変換
            num_bytes2d = token_bytes[y]

            # 特殊トークンを除外して（num_bytes2d > 0）、損失を集計
            total_nats += (loss2d * (num_bytes2d > 0)).sum()

            # バイト数を集計
            total_bytes += num_bytes2d.sum()


    # DDPが有効な場合、全プロセスで合計を集約
    world_size = dist.get_world_size() if dist.is_initialized() else 1
    if world_size > 1:
        dist.all_reduce(total_nats, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_bytes, op=dist.ReduceOp.SUM)

    # 最終的なBPBを計算して返す
    total_nats = total_nats.item()
    total_bytes = total_bytes.item()
    if total_bytes == 0:
        return float('inf')
    
    # BPB = 総損失 / ln(2) / 総バイト数
    bpb = total_nats / (math.log(2) * total_bytes)
    return bpb

In [ ]:
x, y = next(train_loader) # prefetch the very first batch of data

In [ ]:
def save_checkpoint(checkpoint_dir, step, model_data, optimizer_data, meta_data, rank=0):
    if rank == 0:
        os.makedirs(checkpoint_dir, exist_ok=True)
        # Save the model state parameters
        model_path = os.path.join(checkpoint_dir, f"model_{step:06d}.pt")
        torch.save(model_data, model_path)
        logger.info(f"Saved model parameters to: {model_path}")
        # Save the metadata dict as json
        meta_path = os.path.join(checkpoint_dir, f"meta_{step:06d}.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta_data, f, indent=2)
        logger.info(f"Saved metadata to: {meta_path}")
    # Note that optimizer state is sharded across ranks, so each rank must save its own.
    if optimizer_data is not None:
        optimizer_path = os.path.join(checkpoint_dir, f"optim_{step:06d}_rank{rank:d}.pt")
        torch.save(optimizer_data, optimizer_path)
        logger.info(f"Saved optimizer state to: {optimizer_path}")

In [ ]:
logger.setLevel(logging.INFO)

# バイトごとの平均損失（BPB）を初期化 
min_val_bpb = float("inf")

# 訓練損失の指数移動平均を初期化
smooth_train_loss = 0

# 指数移動平均の減衰係数
ema_beta = 0.9

# 訓練のwall-clock timeの合計を初期化
total_training_time = 0

# 訓練のステップカウンターを初期化
step = 0

while True:
    # 総フロップスを計算
    flops_so_far = num_flops_per_token * total_batch_size * step

    # DDPが有効な場合
    if ddp:
        last_step_tensor = torch.tensor(last_step, dtype=torch.int32, device=device)
        dist.all_reduce(last_step_tensor, op=dist.ReduceOp.MAX)
        last_step = bool(last_step_tensor.item())
        logger.info(f"{last_step=}")

    # BPBでモデルを評価
    if eval_every > 0 and (last_step or step % eval_every == 0):
        model.eval()
        val_loader = build_val_loader()
        eval_steps = eval_tokens // (device_batch_size * max_seq_len * ddp_world_size)
        with autocast_ctx:
            val_bpb = evaluate_bpb(model, val_loader, eval_steps, token_bytes)
        logger.info(f"Step {step:05d} | Validation bpb: {val_bpb:.4f} | total training time: {total_training_time/60:.2f}m | total training flops: {flops_so_far/1e15:.2f} PFLOPs")
        if val_bpb < min_val_bpb:
            min_val_bpb = val_bpb
        wandb_run.log({
            "step": step,
            "total_training_flops": flops_so_far,
            "total_training_time": total_training_time,
            "val/bpb": val_bpb,
        })
        model.train()

    # 最後のステップでチェックポイントを保存
    if master_process and last_step and not dry_run:
        output_dirname = f"d{depth}" # e.g. d12
        checkpoint_dir = os.path.join(base_dir, "mid_checkpoints", output_dirname)
        save_checkpoint(
            checkpoint_dir,
            step,
            orig_model.state_dict(),
            [opt.state_dict() for opt in optimizers], # TODO: make sure saving across ranks is done correctly
            {
                "step": step,
                "val_bpb": val_bpb, # loss at last step
                "model_config": {
                    "sequence_len": max_seq_len,
                    "vocab_size": tokenizer.get_vocab_size(),
                    "n_layer": depth,
                    "n_head": model.config.n_head,
                    "n_kv_head": model.config.n_kv_head,
                    "n_embd": model.config.n_embd,
                },
                "user_config": user_config, # inputs to the training script
            }
        )

    # 最後のステップでループを終了
    if last_step:
        break

    # -------------------------------------------------------------------------
    # single training step
    # evaluate the gradient

    synchronize()
    logger.info("Starting training step %d" % step)

    t0 = time.time()

    # 
    for micro_step in range(grad_accum_steps):
        logger.info(f" Micro step {micro_step+1}/{grad_accum_steps}")
        with autocast_ctx:
            loss = model(x, y)
        train_loss = loss.detach() # for logging
        loss = loss / grad_accum_steps # each .backward() is a grad sum => normalize loss here
        loss.backward()
        x, y = next(train_loader) # prefetch the next batch while the GPU is busy with forward/backward
        progress = max(progress, approx_progress) # only increase progress monotonically

    # step the optimizers
    lrm = get_lr_multiplier(progress)
    for opt in optimizers:
        for group in opt.param_groups:
            group["lr"] = group["initial_lr"] * lrm

    muon_momentum = get_muon_momentum(step)

    for group in muon_optimizer.param_groups:
        group["momentum"] = muon_momentum

    for opt in optimizers:
        opt.step()

    model.zero_grad(set_to_none=True)

    synchronize()
    t1 = time.time()
    dt = t1 - t0
    # -------------------------------------------------------------------------

    # State
    step += 1

    # logging
    smooth_train_loss = ema_beta * smooth_train_loss + (1 - ema_beta) * train_loss.item() # EMA the training loss
    debiased_smooth_loss = smooth_train_loss / (1 - ema_beta**(step + 1)) # debias the EMA
    pct_done = 100 * progress
    tok_per_sec = int(world_tokens_per_fwdbwd / dt)
    flops_per_sec = num_flops_per_token * total_batch_size / dt
    promised_flops_per_sec_h100 = 989e12 * ddp_world_size # bfloat16 H100 SXM and without 2:4 sparsity
    mfu = 100 * flops_per_sec / promised_flops_per_sec_h100 # in %

    if step > 10:
        total_training_time += dt # only count the time after the first 10 steps

    logger.info(f"step {step:05d} ({pct_done:.2f}%) | loss: {debiased_smooth_loss:.6f} | lrm: {lrm:.2f} | dt: {dt * 1000:.2f}ms | tok/sec: {tok_per_sec:,} | mfu: {mfu:.2f} | total time: {total_training_time/60:.2f}m")

    if step % 10 == 0:
        wandb_run.log({
            "step": step,
            "total_training_flops": flops_so_far,
            "total_training_time": total_training_time,
            "train/loss": debiased_smooth_loss,
            "train/lrm": lrm,
            "train/dt": dt,
            "train/tok_per_sec": tok_per_sec,
            "train/mfu": mfu,
        })

    break

In [ ]:
logger.setLevel(logging.DEBUG)

## SFT

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import wandb
import torch
import torch.distributed as dist
from contextlib import nullcontext

### ハイパーパラメータの設定

In [ ]:

# -----------------------------------------------------------------------------
# SFT Hyperparameters
run = "dummy" # wandb run name default ("dummy" is special - we won't log to wandb)
# input model options
source = "mid" # base|mid , which checkpoint to load the model from (base model or midtrained model)
model_tag = None # model tag to load the model from (base model or midtrained model)
step = None # step to load the model from (base model or midtrained model)
# compute/precision
device_type = "" # cuda|cpu|mps (empty => autodetect)
dtype = "bfloat16"
device_batch_size = 4 # max to avoid OOM
# optimization
num_epochs = 1

# -1
num_iterations = 1 # override number of iterations (-1 = disable, use num_epochs to derive it)
target_examples_per_step = 32
unembedding_lr = 0.004
embedding_lr = 0.2
matrix_lr = 0.02
weight_decay = 0.0
init_lr_frac = 0.02
# evaluation and logging there of
eval_every = 100
eval_steps = 100
eval_metrics_every = 200
eval_metrics_max_problems = 1024
# now allow CLI to override the settings via the configurator lol
# config_keys = [k for k,v in globals().items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))]
# exec(open(os.path.join('nanochat', 'configurator.py')).read()) # overrides from command line or config file
user_config = {k: globals()[k] for k in config_keys} # possibly useful for logging
# -----------------------------------------------------------------------------
user_config

In [ ]:
# Compute init
device_type = autodetect_device_type() if device_type == "" else device_type
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init(device_type)
master_process = ddp_rank == 0
ptdtype = torch.float32 if dtype == 'float32' else torch.bfloat16
autocast_ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype) if device_type == "cuda" else nullcontext()

In [ ]:
# wandb logging init
use_dummy_wandb = run == "dummy" or not master_process
wandb_run = DummyWandb() if use_dummy_wandb else wandb.init(project="nanochat-sft", name=run, config=user_config, save_code=True)

In [ ]:
def find_last_step(checkpoint_dir):
    # Look into checkpoint_dir and find model_<step>.pt with the highest step
    checkpoint_files = glob.glob(os.path.join(checkpoint_dir, "model_*.pt"))
    if not checkpoint_files:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
    last_step = int(max(os.path.basename(f).split("_")[-1].split(".")[0] for f in checkpoint_files))
    return last_step

In [ ]:
def find_largest_model(checkpoint_dir):
    # attempt to guess the model tag: take the biggest model available
    model_tags = [f for f in os.listdir(checkpoint_dir) if os.path.isdir(os.path.join(checkpoint_dir, f))]
    if not model_tags:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")
    # 1) normally all model tags are of the form d<number>, try that first:
    candidates = []
    for model_tag in model_tags:
        match = re.match(r"d(\d+)", model_tag)
        if match:
            model_depth = int(match.group(1))
            candidates.append((model_depth, model_tag))
    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1]
    # 2) if that failed, take the most recently updated model:
    model_tags.sort(key=lambda x: os.path.getmtime(os.path.join(checkpoint_dir, x)), reverse=True)
    return model_tags[0]

In [ ]:
# Load the model and tokenizer
model, tokenizer, meta = load_model(source, device, phase="train", model_tag=model_tag, step=step)
orig_model = model # original, uncompiled model
# model = torch.compile(model, dynamic=True) # doesn't work super well because of variable lengths of inputs
engine = Engine(model, tokenizer) # will be used for inline model evaluation only

In [ ]:
class ARC(Task):
    """
    The ARC dataset from Allen AI.
    https://huggingface.co/datasets/allenai/ai2_arc
    """

    def __init__(self, subset, split, **kwargs):
        super().__init__(**kwargs)
        assert subset in ["ARC-Easy", "ARC-Challenge"], "ARC subset must be ARC-Easy or ARC-Challenge"
        assert split in ["train", "validation", "test"], "ARC split must be train|validation|test"
        self.ds = load_dataset("allenai/ai2_arc", subset, split=split).shuffle(seed=42)

    @property
    def eval_type(self):
        return 'categorical'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        row = self.ds[index]
        question = row["question"] # the question text
        choices = row["choices"]["text"] # the text of each choice
        answer_string = row["answerKey"] # e.g. "A", "B", "C", "D"
        letters = row["choices"]["label"] # e.g. ["A", "B", "C", "D"]
        assert answer_string in letters, f"ARC answer {answer_string} must be one of {letters}" # sanity check
        # create and return the Conversation object
        user_message = render_mc(question, letters, choices)
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer_string}
        ]
        conversation = {
            "messages": messages,
            "letters": letters, # useful during evaluation, so we can narrow and clamp the assistant prediction to one of the letters
        }
        return conversation

    def evaluate(self, conversation, assistant_response):
        # the assert here is not strictly speaking needed, but currently the way we eval, we expect this to be true
        # I'm going to leave the assert here to prevent footguns, but possibly in the future can remove it.
        assert assistant_response in conversation['letters'], f"ARC answer {assistant_response} is expected to be one of {conversation['letters']}"
        assistant_message = conversation['messages'][-1]['content'] # e.g. "A"
        return assistant_response == assistant_message

In [ ]:
# Task data mixture we'll train on
identity_conversations_filepath = os.path.join(get_base_dir(), "identity_conversations.jsonl")
train_ds = TaskMixture([
    ARC(subset="ARC-Easy", split="train"), # 2.3K rows
    ARC(subset="ARC-Challenge", split="train"), # 1.1K rows
    GSM8K(subset="main", split="train"), # 8K rows
    SmolTalk(split="train", stop=10_000), # 10K rows of smoltalk
    CustomJSON(filepath=identity_conversations_filepath), # 1K rows of synthetic identity conversations
    SimpleSpelling(size=300, split="train"), # 300 rows of Simple Spelling (e.g. spell the word 'apple')
    SpellingBee(size=300, split="train"), # 300 rows of Spelling Bee (e.g. how many 'r' are in 'strawberry'?)
]) # 2.3K + 1.1K + 8K + 10K + 1K + 0.3K + 0.3K = 23K rows

In [ ]:
val_ds = SmolTalk(split="test") # general conversations, 24K rows (though we don't actually use all of it)

### データローダー

In [ ]:
def sft_data_generator(dataset, batch_size):
    pad_token_id = tokenizer.encode_special("<|assistant_end|>") # use <|assistant_end|> as the pad token is ok, these positions are masked in the loss
    # prepares a list of tokenized conversations into a batch and yields
    def collate_and_yield(batch):
        nrows = len(batch)
        ncols = max(len(ids) for ids, mask in batch) - 1 # seq of n creates inputs/targets of n-1
        inputs = torch.full((nrows, ncols), pad_token_id, dtype=torch.long)
        targets = torch.full((nrows, ncols), -1, dtype=torch.long) # -1 is ignore index
        for i, (ids, mask) in enumerate(batch):
            n = len(ids)
            ids_tensor = torch.tensor(ids, dtype=torch.long)
            inputs[i, :n-1] = ids_tensor[:-1]
            # recall -1 is the ignore index, so mask out targets where mask is 0
            row_targets = ids_tensor[1:]
            # mask[1:] omits the mask for the BOS token, which is never a target atm so it's ok
            mask_tensor = torch.tensor(mask[1:], dtype=torch.long)
            row_targets[mask_tensor == 0] = -1 # mask out targets where mask is 0
            targets[i, :n-1] = row_targets
        inputs = inputs.to(device) # move to device
        targets = targets.to(device)
        return inputs, targets
    # iterates over the dataset in epochs, tokenizes
    batch = []
    while True:
        for i in range(ddp_rank, len(dataset), ddp_world_size):
            doc = dataset[i]
            ids, mask = tokenizer.render_conversation(doc)
            batch.append((ids, mask))
            if len(batch) == batch_size:
                yield collate_and_yield(batch)
                batch = []

In [ ]:
examples_per_step = device_batch_size * ddp_world_size
logger.debug(f"Target examples per step: {target_examples_per_step}")
logger.debug(f"Device batch size: {device_batch_size}")
logger.debug(f"Examples per step is device_batch_size * ddp_world_size: {examples_per_step}")
assert target_examples_per_step % examples_per_step == 0, "Target examples per step must be divisible by examples per step"
grad_accum_steps = target_examples_per_step // examples_per_step
logger.debug(f"=> Setting grad accum steps: {grad_accum_steps}")

In [ ]:
if num_iterations == -1:
    # derive num_iterations from num_epochs and the size of the dataset
    assert num_epochs > 0, "num_epochs must be positive if num_iterations is -1"
    num_iterations = (len(train_ds) // target_examples_per_step) * num_epochs
    logger.debug(f"Derived num_iterations from num_epochs: {num_iterations}")

In [ ]:
train_loader = sft_data_generator(train_ds, batch_size=device_batch_size)
build_val_loader = lambda: sft_data_generator(val_ds, batch_size=device_batch_size)

In [ ]:
# Initialize the Optimizer

optimizers = model.setup_optimizers(
    unembedding_lr=unembedding_lr,
    embedding_lr=embedding_lr,
    matrix_lr=matrix_lr,
    weight_decay=weight_decay,
)

In [ ]:
for opt in optimizers:
    for group in opt.param_groups:
        group["lr"] = group["lr"] * init_lr_frac
        group["initial_lr"] = group["lr"] # save the initial learning so we can decay easily later

In [ ]:
# Learning rate scheduler
def get_lr_multiplier(it):
    lrm = 1.0 - it / num_iterations
    return lrm

### 評価

In [ ]:
def extract_imports(prompt):
    """Extract import statements from the beginning of a code block."""
    imports = []
    for line in prompt.split('\n'):
        stripped = line.strip()
        if stripped.startswith('import ') or stripped.startswith('from '):
            imports.append(stripped)
        elif stripped and not stripped.startswith('#'):
            # Stop at first non-import, non-comment line
            break
    return '\n'.join(imports)

In [ ]:
def extract_program(completion):
    """
    Extract Python code from LLM completion.

    Handles various output formats:
    - Code wrapped in ```python ... ``` or ``` ... ``` blocks
    - Plain code without markdown blocks
    - Extra text before/after code blocks

    Returns the first code block if found, otherwise returns the whole completion.
    """
    # Try to find markdown code blocks (```python or just ```)
    # Match ```python\n...\n``` or ```\n...\n```
    pattern = r'```(?:python)?\s*\n(.*?)\n```'
    matches = re.findall(pattern, completion, re.DOTALL)

    if matches:
        # Return the first code block found
        return matches[0].strip()

    # No code blocks found, return the whole completion
    return completion.strip()


In [ ]:
class HumanEval(Task):

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.ds = load_dataset("openai/openai_humaneval", split="test").shuffle(seed=42)

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        """ Get a single problem from the dataset. """
        row = self.ds[index]
        prompt = row['prompt'] # prompts in HumanEval are the beginning of the program
        solution = row['canonical_solution'] # the correct continuation of the program
        entry_point = row['entry_point'] # the function to check
        test = row['test'] # the test cases
        complete_solution = f"{prompt}\n{solution}"
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": complete_solution},
        ]
        conversation = {
            "messages": messages,
            "entry_point": entry_point, # needed during evaluation
            "test": test, # needed during evaluation
        }
        return conversation

    def evaluate(self, conversation, completion):
        """ Given (conversation, completion), return boolean success of the completion. """
        # the prompt will contain the imports and the function signature
        imports = extract_imports(conversation['messages'][0]['content'])
        # the completion will usually contain the whole function
        # but not always with the needed imports, so we manually append them
        completion_code = extract_program(completion)
        program = (
            imports
            + "\n\n"
            + completion_code
            + "\n\n"
            + conversation['test']
            + "\n"
            + f"check({conversation['entry_point']})"
        )
        result = execute_code(program)
        success = result.success
        return success

In [ ]:
# Generative evaluation loop (we go one problem at a time, sample, evaluate)

def run_generative_eval(task_object, tokenizer, model, engine, num_samples, max_new_tokens, temperature, top_k, max_problems=None):

    ddp, ddp_rank, ddp_local_rank, ddp_world_size = get_dist_info()
    device = model.get_device()

    num_problems = len(task_object) if max_problems is None else min(len(task_object), max_problems)

    # Run the evaluation
    num_passed, total = 0, 0
    for i in range(ddp_rank, num_problems, ddp_world_size):
        conversation = task_object[i]

        # Tokenize the prompt
        encoded_prompt = tokenizer.render_for_completion(conversation)
        # Get the completions
        results, _ = engine.generate_batch(
            encoded_prompt,
            num_samples=num_samples,
            max_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )
        # Decode the completions as text
        prefix_length = len(encoded_prompt)
        completions = [tokenizer.decode(result_tokens[prefix_length:]) for result_tokens in results]
        # Evaluate success criteria
        outcomes = [task_object.evaluate(conversation, completion) for completion in completions]
        passed = any(outcomes)

        # Keep stats
        total += 1
        num_passed += int(passed)

        # Logging (overwrite the same line in the console)
        print(f"\r\033[KRank {ddp_rank} | {num_passed}/{total} ({100*num_passed/total:.2f}%)", end='', flush=True)

    # Finish the in-place progress line with a newline before final summary
    print()

    # Aggregate results across all ranks
    if ddp:
        num_passed_tensor = torch.tensor([num_passed], dtype=torch.long, device=device)
        total_tensor = torch.tensor([total], dtype=torch.long, device=device)
        dist.all_reduce(num_passed_tensor, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_tensor, op=dist.ReduceOp.SUM)
        num_passed = num_passed_tensor.item()
        total = total_tensor.item()

    logger.debug("=" * 50)
    logger.debug(f"Final: {num_passed}/{total} ({100*num_passed/total:.2f}%)")

    # Return the accuracy
    return num_passed/total

In [ ]:
# Categorical evaluation loop
# A lot easier because we don't have to sample. Therefore, we can actually go
# batches at a time and just check the logits for correct answer choices.

def run_categorical_eval(task_object, tokenizer, model, batch_size, max_problems=None):

    ddp, ddp_rank, ddp_local_rank, ddp_world_size = get_dist_info()
    device = model.get_device()
    bos = tokenizer.get_bos_token_id() # use BOS as pad token is ok, these positions are ignored

    # We'll process batches of independent problems at a time because there is no sampling needed
    num_problems = len(task_object) if max_problems is None else min(len(task_object), max_problems)
    ceil_div = lambda x, y: -(-x // y)
    num_batches = ceil_div(num_problems, batch_size)

    # Run the evaluation
    letter_to_id_cache = {} # many letters will repeat often, let's save the tokenizer some work
    num_passed, total = 0, 0
    for i in range(ddp_rank, num_batches, ddp_world_size):
        i0, i1 = i * batch_size, min((i + 1) * batch_size, num_problems)

        # Prepare the batch of problems. They might all be of different length, so we pad/collate them.
        conversations = [task_object[ii] for ii in range(i0, i1)]
        prompt_ids = [tokenizer.render_for_completion(conversation) for conversation in conversations] # TODO: remake the way this works
        max_length = max(len(ids) for ids in prompt_ids)
        answer_time_positions = [len(ids) - 1 for ids in prompt_ids] # where the last token is (and the predicted answer)
        padded_prompt_ids = [ids + [bos] * (max_length - len(ids)) for ids in prompt_ids]
        prompt_ids = torch.tensor(padded_prompt_ids, dtype=torch.long, device=device)

        # Get the logits for the whole batch of conversations in parallel (efficiency win here)
        with torch.no_grad():
            logits = model(prompt_ids) # (B, T, V)

        # Focus on the available answer on just the letters corresponding to choices
        # Note that this helps the evaluation a lot because it specifically narrows the focus to only the available letters
        # The much harder alternative would be to just generate from the Assistant and check if it responded with the correct
        # letter (e.g. A, B, C, D), but evaluations typically make the task easier in this way.
        for idx, conversation in enumerate(conversations):
            # get the token ids of all the available letters of this problem
            letters = conversation['letters']
            letter_ids = []
            for letter in letters:
                if not letter in letter_to_id_cache:
                    encoded_letter = tokenizer.encode(letter)
                    assert len(encoded_letter) == 1, "Each letter must be a single token"
                    letter_to_id_cache[letter] = encoded_letter[0]
                letter_ids.append(letter_to_id_cache[letter])
            # focus logits just down to the answer position and the available letters of the answer
            answer_pos = answer_time_positions[idx]
            focus_logits = logits[idx, answer_pos, letter_ids]
            # get the argmax letter (the predicted answer)
            argmax_letter_id = focus_logits.argmax(dim=-1).item()
            predicted_letter = letters[argmax_letter_id]
            # evaluate the outcome
            outcome = task_object.evaluate(conversation, predicted_letter)
            num_passed += int(outcome)
            total += 1

    # Aggregate results across all ranks
    if ddp:
        num_passed_tensor = torch.tensor([num_passed], dtype=torch.long, device=device)
        total_tensor = torch.tensor([total], dtype=torch.long, device=device)
        dist.all_reduce(num_passed_tensor, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_tensor, op=dist.ReduceOp.SUM)
        num_passed = num_passed_tensor.item()
        total = total_tensor.item()

    average = num_passed/total
    logger.debug(f"Final: {num_passed}/{total} ({100*average:.2f}%)")
    return average


In [ ]:
def run_chat_eval(task_name, model, tokenizer, engine,
                   batch_size=1, num_samples=1, max_new_tokens=512, temperature=0.0, top_k=50,
                   max_problems=None):
    # Create the evaluation object
    task_module = {
        'HumanEval': HumanEval,
        'MMLU': partial(MMLU, subset="all", split="test"),
        'ARC-Easy': partial(ARC, subset="ARC-Easy", split="test"),
        'ARC-Challenge': partial(ARC, subset="ARC-Challenge", split="test"),
        'GSM8K': partial(GSM8K, subset="main", split="test"),
        'SpellingBee': partial(SpellingBee, size=256, split="test"),
    }[task_name]
    task_object = task_module()
    # Run the evaluation
    if task_object.eval_type == 'generative':
        acc = run_generative_eval(task_object, tokenizer, model, engine, num_samples, max_new_tokens, temperature, top_k, max_problems=max_problems)
    elif task_object.eval_type == 'categorical':
        acc = run_categorical_eval(task_object, tokenizer, model, batch_size, max_problems=max_problems)
    else:
        raise ValueError(f"Unsupported task evaluation type: {task_object.eval_type}")
    return acc

### 訓練

In [ ]:
logger.setLevel(logging.INFO)

# Go!
step = 0
train_iter = iter(train_loader)
metrics = {}
for step in range(num_iterations):
    last_step = step == num_iterations - 1

    # evaluate the validation loss
    if last_step or step % eval_every == 0:
        model.eval()
        val_iter = iter(build_val_loader())
        losses = []
        for _ in range(eval_steps):
            val_inputs, val_targets = next(val_iter)
            with torch.no_grad(), autocast_ctx:
                loss = model(val_inputs, val_targets)
            losses.append(loss)
        val_loss = torch.stack(losses).mean() # average over eval_steps
        if ddp:
            dist.all_reduce(val_loss, op=dist.ReduceOp.AVG) # average over ranks
        val_loss = val_loss.item()
        logger.info(f"Step {step:05d} | Validation loss: {val_loss:.6f}")
        wandb_run.log({
            "step": step,
            "val_loss": val_loss,
        })
        model.train()

    # evaluate accuracy of the multiple choice tasks (which are quick to run)
    if last_step or (step > 0 and step % eval_metrics_every == 0):
        model.eval()
        with torch.no_grad(), autocast_ctx:
            # note that because these are inside no_grad, we can usually afford to at least ~2X the batch size
            metrics["mmlu_acc"] = run_chat_eval("MMLU", model, tokenizer, engine, batch_size=device_batch_size*2, max_problems=eval_metrics_max_problems)
            metrics["arc_easy_acc"] = run_chat_eval("ARC-Easy", model, tokenizer, engine, batch_size=device_batch_size*2, max_problems=eval_metrics_max_problems)
        metrics_str = ', '.join(f'{k}: {v:.6f}' for k, v in metrics.items())
        logger.info(f"Step {step:05d} | {metrics_str}")
        wandb_run.log({
            "step": step,
            **metrics,
        })
        model.train()

    if last_step:
        break

    # evaluate the gradient
    num_tokens = torch.tensor(0, device=device) # the number of "active" tokens of supervision seen
    for micro_step in range(grad_accum_steps):
        logger.info(f" Step {step:05d} | Micro step {micro_step+1}/{grad_accum_steps}")
        train_inputs, train_targets = next(train_iter)
        with autocast_ctx:
            loss = model(train_inputs, train_targets)
        train_loss = loss.detach() # for logging
        loss = loss / grad_accum_steps # each .backward() is a grad sum => normalize loss here
        loss.backward() # accumulate the gradient
        num_tokens += (train_targets >= 0).sum()

    if ddp:
        dist.all_reduce(num_tokens, op=dist.ReduceOp.SUM) # sum over ranks

    # learning rate scheduler
    lrm = get_lr_multiplier(step)
    for opt in optimizers:
        for group in opt.param_groups:
            group["lr"] = group["initial_lr"] * lrm

    # step the optimizers
    for opt in optimizers:
        opt.step()

    model.zero_grad(set_to_none=True)

    # logging
    train_loss_item = train_loss.item()
    num_tokens_item = num_tokens.item()
    logger.info(f"Step {step:05d}/{num_iterations:05d} | Training loss: {train_loss_item:.6f}| lrm: {lrm:.6f}| num_tokens: {num_tokens_item:,}")

    wandb_run.log({
        "step": step,
        "lrm": lrm,
        "train_loss": train_loss_item,
        "num_tokens": num_tokens_item,
    })
    step += 1

    break

logger.setLevel(logging.DEBUG)

In [ ]:
# Save the model at the end of the run
if master_process:
    base_dir = get_base_dir()
    depth = model.config.n_layer
    model_tag = f"d{depth}" # base the model tag on the depth of the base model
    checkpoint_dir = os.path.join(base_dir, "chatsft_checkpoints", model_tag)
    model_config_kwargs = model.config.__dict__ # slightly naughty, abusing the simplicity of GPTConfig, TODO nicer
    save_checkpoint(
        checkpoint_dir,
        step,
        model.state_dict(),
        None, # note: we don't bother to save the optimizer state
        {
            "step": step,
            "val_loss": val_loss,
            **metrics,
            "model_config": model_config_kwargs,
        }
    )
    print(f"✅ Saved model checkpoint to {checkpoint_dir}")

In [ ]:
# Log to report
# from nanochat.report import get_report
# get_report().log(section="Chat SFT", data=[
#     user_config, # CLI args
#     {
#         "Training rows": len(train_ds),
#         "Number of iterations": num_iterations,
#         "Training loss": train_loss_item,
#         "Validation loss": val_loss,
#     },
# ])

In [ ]:
# Cleanup
wandb_run.finish()

def compute_cleanup():
    """Companion function to compute_init, to clean things up before script exit"""
    if is_ddp():
        dist.destroy_process_group()

compute_cleanup()

## 強化学習

In [ ]:
"""
Reinforcement learning on GSM8K via "GRPO".

I put GRPO in quotes because we actually end up with something a lot
simpler and more similar to just REINFORCE:

1) Delete trust region, so there is no KL regularization to a reference model
2) We are on policy, so there's no need for PPO ratio+clip.
3) We use GAPO style normalization that is token-level, not sequence-level.
4) Instead of z-score normalization (r - mu)/sigma, only use (r - mu) as the advantage.

1 GPU:
python -m scripts.chat_rl

8 GPUs:
torchrun --standalone --nproc_per_node=8 -m scripts.chat_rl -- --run=default
"""

import os
import itertools
import re
import wandb
import torch
import torch.distributed as dist

### ハイパーパラメータ

In [ ]:

# RL hyperparameters
run = "dummy" # wandb run name
source = "sft" # mid|sft
dtype = "bfloat16"
device_batch_size = 8 # no forward pass will go above this to not OOM
examples_per_step = 16 # in total and across all ranks (note: examples, not samples/completions!)
num_samples = 16 # number of samples per example (/question)
max_new_tokens = 256
temperature = 1.0
top_k = 50 # TODO: try None?
unembedding_lr = 0.004
embedding_lr = 0.2
matrix_lr = 0.02
weight_decay = 0.0
init_lr_frac = 0.05
num_epochs = 1 # how many epochs of gsm8k to train on
save_every = 60 # every how many steps to save the model
eval_every = 60 # every how many steps to evaluate the model for val pass@k
eval_examples = 400 # number of examples used for evaluating pass@k
# now allow CLI to override the settings via the configurator lol
config_keys = [k for k,v in globals().items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))]
# exec(open(os.path.join('nanochat', 'configurator.py')).read()) # overrides from command line or config file
user_config = {k: globals()[k] for k in config_keys} # will be useful for logging
user_config

In [ ]:
# Init compute/precision
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init()
master_process = ddp_rank == 0 # this process will do logging, checkpointing etc.
dtype = torch.float32 if dtype == 'float32' else torch.bfloat16
autocast_ctx = torch.amp.autocast(device_type="cuda", dtype=dtype)


In [ ]:
# wandb logging init
use_dummy_wandb = run == "dummy" or not master_process
wandb_run = DummyWandb() if use_dummy_wandb else wandb.init(project="nanochat-rl", name=run, config=user_config)

In [ ]:
# Init model and tokenizer
model, tokenizer, meta = load_model(source, device, phase="eval")
engine = Engine(model, tokenizer) # for sampling rollouts

In [ ]:
# Rollout / sampling generator loop that yields batches of examples for training

train_task = GSM8K(subset="main", split="train")
val_task = GSM8K(subset="main", split="test")
num_steps = (len(train_task) // examples_per_step) * num_epochs
logger.info(f"Calculated number of steps: {num_steps}")

In [ ]:
@torch.no_grad()
def get_batch():
    assistant_end = tokenizer.encode_special("<|assistant_end|>") # ok to use this token, it's only for padding and isn't used in the loss.
    rank_indices = range(ddp_rank, len(train_task), ddp_world_size) # each rank is responsible for different examples in the training data
    for example_idx in itertools.cycle(rank_indices):

        # First get the full conversation of both user and assistant messages
        conversation = train_task[example_idx]

        # Tokenize the conversation, deleting the last Assistant message and priming the Assistant for a completion instead
        # (i.e. keep the <|assistant_start|>, but delete everything after it)
        tokens = tokenizer.render_for_completion(conversation)
        prefix_length = len(tokens)

        # Generate num_samples samples using batched generation, use loop to avoid OOMs
        model.eval() # ensure the model is in eval mode
        generated_token_sequences = []
        masks = []
        num_sampling_steps = num_samples // device_batch_size # go sequentially to prevent OOMs
        for sampling_step in range(num_sampling_steps):
            seed = hash((step, example_idx, sampling_step)) & 0x7FFFFFFF # positive half of int32
            with autocast_ctx:
                generated_token_sequences_batch, masks_batch = engine.generate_batch(
                    tokens,
                    num_samples=device_batch_size,
                    max_tokens=max_new_tokens,
                    temperature=temperature,
                    top_k=top_k,
                    seed=seed, # must make sure to change the seed for each sampling step
                )
            generated_token_sequences.extend(generated_token_sequences_batch)
            masks.extend(masks_batch)

        # Calculate the rewards for each sample
        rewards = []
        for sample_tokens in generated_token_sequences:
            # Get just the generated tokens (after the prompt)
            generated_tokens = sample_tokens[prefix_length:]
            # Decode the generated response
            generated_text = tokenizer.decode(generated_tokens)
            # Calculate the reward
            reward = train_task.reward(conversation, generated_text)
            rewards.append(reward)

        # Pad the sequences so that their lengths (in time) match
        max_length = max(len(seq) for seq in generated_token_sequences)
        padded_generated_token_sequences = [seq + [assistant_end] * (max_length - len(seq)) for seq in generated_token_sequences]
        padded_masks = [mask + [0] * (max_length - len(mask)) for mask in masks]
        # Stack up the sequences and masks into PyTorch tensors
        ids = torch.tensor(padded_generated_token_sequences, dtype=torch.long, device=device)
        mask_ids = torch.tensor(padded_masks, dtype=torch.long, device=device)
        # Generate autoregressive inputs and targets to the Transformer
        inputs = ids[:, :-1]
        targets = ids[:, 1:].clone() # clone to avoid in-place modification:
        targets[mask_ids[:, 1:] == 0] = -1 # <-- inplace modification right here. -1 is the ignore index
        # NOTE also that the Engine returns mask=0 for BOTH the prompt tokens AND the tool use tokens.
        # So we will (correctly) end up not training on the prompt tokens, or the tool use forced tokens.
        rewards = torch.tensor(rewards, dtype=torch.float, device=device)
        # Calculate the advantages by simply subtracting the mean (instead of z-score (x-mu)/sigma)
        mu = rewards.mean()
        advantages = rewards - mu
        # yield inputs/targets as (B, T) of ids and rewards as (B,) of floats
        yield generated_token_sequences, inputs, targets, rewards, advantages

In [ ]:
# Simple evaluation loop for GSM8K pass@k
def run_gsm8k_eval(task, tokenizer, engine,
    max_examples=None,
    num_samples=1,
    max_completion_tokens=256,
    temperature=0.0,
    top_k=50
):
    """
    Evaluates GSM8K task and returns a list of records of evaluation outcomes.
    In a distributed setting, all ranks cooperate but this function will NOT
    do the reduction across ranks. This is the responsibility of the caller.
    Because the evaluation can take a while, this function will yield records one by one.
    """
    max_examples = min(max_examples, len(task)) if max_examples is not None else len(task)
    for idx in range(ddp_rank, max_examples, ddp_world_size):
        conversation = task[idx]
        tokens = tokenizer.render_for_completion(conversation)
        prefix_length = len(tokens)
        # Generate k samples using batched generation inside the Engine
        assert num_samples <= device_batch_size # usually this is true. we can add a loop if not...
        generated_token_sequences, masks = engine.generate_batch(
            tokens,
            num_samples=num_samples,
            max_tokens=max_completion_tokens,
            temperature=temperature,
            top_k=top_k
        )
        # Check each sample for correctness
        outcomes = []
        for sample_tokens in generated_token_sequences:
            generated_tokens = sample_tokens[prefix_length:]
            generated_text = tokenizer.decode(generated_tokens)
            is_correct = task.evaluate(conversation, generated_text)
            outcomes.append({
                "is_correct": is_correct
            })
        # A bit bloated because I wanted to do more complex logging at one point.
        record = {
            "idx": idx,
            "outcomes": outcomes,
        }
        yield record

In [ ]:
# Init the optimizer
optimizers = model.setup_optimizers(
    unembedding_lr=unembedding_lr,
    embedding_lr=embedding_lr,
    matrix_lr=matrix_lr,
    weight_decay=weight_decay,
)

In [ ]:

# Set the initial learning rate as a fraction of the base learning rate
for opt in optimizers:
    for group in opt.param_groups:
        group["lr"] = group["lr"] * init_lr_frac
        group["initial_lr"] = group["lr"] # save the initial learning so we can decay easily later

In [ ]:
# Learning rate scheduler: simple rampdown to zero over num_steps
def get_lr_multiplier(it):
    lrm = 1.0 - it / num_steps
    return lrm

In [ ]:
# Calculate the number of examples each rank handles to achieve the desired examples_per_step
logger.info(f"Total sequences per step: {examples_per_step * num_samples}") # total batch size in sequences/step
assert examples_per_step % ddp_world_size == 0, "Desired examples per step must be divisible by the number of ranks"
examples_per_rank = examples_per_step // ddp_world_size # per GPU
logger.info(f"Calculated examples per rank: {examples_per_rank}")

In [ ]:
logger.setLevel(logging.INFO)

# Kick off the training loop
batch_iterator = get_batch()
for step in range(num_steps):

    # Evaluate the model once in a while and log to wandb
    if step % eval_every == 0:
        model.eval()
        passk = torch.zeros(device_batch_size, device=device) # pass@k for k=1..device_batch_size

        with autocast_ctx:
            records_iter = run_gsm8k_eval(val_task, tokenizer, engine, num_samples=device_batch_size, max_examples=eval_examples, temperature=1.0)
            records = list(records_iter) # collect all records

        for k in range(1, device_batch_size + 1):
            passk[k - 1] = sum(any(o["is_correct"] for o in r["outcomes"][:k]) for r in records)

        num_records = torch.tensor(len(records), dtype=torch.long, device=device)

        if ddp:
            dist.all_reduce(num_records, op=dist.ReduceOp.SUM)
            dist.all_reduce(passk, op=dist.ReduceOp.SUM)

        passk = passk / num_records.item() # normalize by the total number of records
        print_passk = [f"Pass@{k}: {passk[k - 1].item():.4f}" for k in range(1, device_batch_size + 1)]
        logger.info(f"Step {step} | {', '.join(print_passk)}")
        log_passk = {f"pass@{k}": passk[k - 1].item() for k in range(1, device_batch_size + 1)}
        wandb_run.log({
            "step": step,
            **log_passk,
        })

    # Forward/Backward on rollouts over multiple examples in the dataset
    rewards_list = []
    sequence_lengths = []
    for example_step in range(examples_per_rank):
        # Get one batch corresponding to one example in the training dataset
        sequences_all, inputs_all, targets_all, rewards_all, advantages_all = next(batch_iterator)
        # Evaluate the loss and gradients
        model.train() # ensure the model is in train mode
        # We need one more loop because we can never exceed the device_batch_size
        assert inputs_all.size(0) % device_batch_size == 0
        num_passes = inputs_all.size(0) // device_batch_size
        for pass_idx in range(num_passes):
            # Pluck out the batch for this pass
            b0, b1 = pass_idx * device_batch_size, (pass_idx + 1) * device_batch_size
            inputs = inputs_all[b0:b1]
            targets = targets_all[b0:b1]
            rewards = rewards_all[b0:b1]
            advantages = advantages_all[b0:b1]
            # Calculate log probabilities. Note that the loss calculates NLL = -logp, so we negate
            with autocast_ctx:
                logp = -model(inputs, targets, loss_reduction='none').view_as(inputs) # (B, T)
            # Calculate the PG objective. Note that ignore_index=-1 ensures that invalid tokens have loss 0.
            pg_obj = (logp * advantages.unsqueeze(-1)).sum()
            # normalize by the number of valid tokens, number of passes, and examples_per_rank
            num_valid = (targets >= 0).sum().clamp(min=1)
            pg_obj = pg_obj / (num_valid * num_passes * examples_per_rank)
            # Note, there is no need to add PPO ratio+clip because we are on policy
            # Finally, formulate the loss that we want to minimize (instead of objective we wish to maximize)
            loss = -pg_obj
            loss.backward()
            logger.info(f"Step {step}/{num_steps} | Example step {example_step} | Pass {pass_idx} | loss: {loss.item():.6f} | Average reward: {rewards.mean().item()}")
        # For logging
        rewards_list.append(rewards_all.mean().item())
        sequence_lengths.extend(len(seq) for seq in sequences_all)

    # A bunch of logging for how the rollouts went this step
    mean_reward = sum(rewards_list) / len(rewards_list)
    mean_sequence_length = sum(sequence_lengths) / len(sequence_lengths)
    if ddp: # aggregate across ranks
        mean_reward_tensor = torch.tensor(mean_reward, dtype=torch.float, device=device)
        mean_sequence_length_tensor = torch.tensor(mean_sequence_length, dtype=torch.float, device=device)
        dist.all_reduce(mean_reward_tensor, op=dist.ReduceOp.AVG)
        dist.all_reduce(mean_sequence_length_tensor, op=dist.ReduceOp.AVG)
        mean_reward = mean_reward_tensor.item()
        mean_sequence_length = mean_sequence_length_tensor.item()
    logger.info(f"Step {step}/{num_steps} | Average reward: {mean_reward} | Average sequence length: {mean_sequence_length:.2f}")
    wandb_run.log({
        "step": step,
        "reward": mean_reward,
        "sequence_length": mean_sequence_length,
    })

    # Update the model parameters
    lrm = get_lr_multiplier(step)
    for opt in optimizers: # first set the learning rate
        for group in opt.param_groups:
            group["lr"] = group["initial_lr"] * lrm
    for opt in optimizers: # then step the optimizers
        opt.step()
    model.zero_grad(set_to_none=True)
    wandb_run.log({
        "step": step,
        "lrm": lrm,
    })

    # Master process saves the model once in a while. Skip first step. Save last step.
    if master_process and ((step > 0 and step % save_every == 0) or step == num_steps - 1):
        base_dir = get_base_dir()
        depth = model.config.n_layer
        model_tag = f"d{depth}" # base the model tag on the depth of the base model
        checkpoint_dir = os.path.join(base_dir, "chatrl_checkpoints", model_tag)
        model_config_kwargs = model.config.__dict__ # slightly naughty, abusing the simplicity of GPTConfig, TODO nicer
        save_checkpoint(
            checkpoint_dir,
            step,
            model.state_dict(),
            None, # note: we don't bother to save the optimizer state
            {
                "model_config": model_config_kwargs,
            }
        )
        logger.info(f"✅ Saved model checkpoint to {checkpoint_dir}")

    break

logger.setLevel(logging.DEBUG)